In [ ]:
# Importar las librerías necesarias
import pandas as pd
import numpy as np
import sweetviz as sv
import seaborn as sns
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
import warnings
from scipy.stats import zscore, mode
from sklearn.metrics import (silhouette_samples,silhouette_score,make_scorer,mean_absolute_error, r2_score, mean_squared_error,accuracy_score,precision_score,recall_score,f1_score,roc_auc_score)
from sklearn.base import (BaseEstimator,TransformerMixin,ClassifierMixin,RegressorMixin)
from sklearn.pipeline import Pipeline
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, BaseCrossValidator, KFold, cross_val_score, RandomizedSearchCV, GridSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
import joblib
from joblib import load
import os
from tslearn.preprocessing import TimeSeriesScalerMeanVariance
from tslearn.clustering import TimeSeriesKMeans, silhouette_score as ts_silhouette_score
import shap
from lime import lime_tabular
from sklearn.model_selection import GroupKFold

In [ ]:
# Cargar los datos desde el archivo Excel
file_path = 'C:/Users/Andy/OneDrive/Desktop/MCD/Tesis/Datos_fuente_Bloomberg/en valores/serie completa 2014-2024/Dataset/dataset_completo.xlsx'
df = pd.read_excel(file_path, sheet_name='dataset')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# Quitar "k" de las columnas "Non farm payrolls_Exp_mediana" y "Non farm payrolls_Exp_promedio"
df['Non farm payrolls_Exp_mediana'] = df['Non farm payrolls_Exp_mediana'].astype(str).str.replace('k', '')
df['Non farm payrolls_Exp_promedio'] = df['Non farm payrolls_Exp_promedio'].astype(str).str.replace('k', '')

# Convertir las columnas a valores numéricos
df['Non farm payrolls_Exp_mediana'] = pd.to_numeric(df['Non farm payrolls_Exp_mediana'], errors='coerce')
df['Non farm payrolls_Exp_promedio'] = pd.to_numeric(df['Non farm payrolls_Exp_promedio'], errors='coerce')
df['P_B'] = pd.to_numeric(df['P_B'], errors='coerce')
df['P_TB'] = pd.to_numeric(df['P_TB'], errors='coerce')
df['P_S'] = pd.to_numeric(df['P_S'], errors='coerce')
df['P_FCF'] = pd.to_numeric(df['P_FCF'], errors='coerce')
df['P_Share'] = pd.to_numeric(df['P_Share'], errors='coerce')

In [ ]:
# Convertir la columna de fecha a formato datetime
df['Fecha'] = pd.to_datetime(df['Fecha'], format='%Y%m%d')

# Extraer características temporales relevantes
df['mes'] = df['Fecha'].dt.month
df['año'] = df['Fecha'].dt.year
df['periodo'] = df['Fecha'].dt.to_period('M')

In [ ]:
# Convertir CPI y Fed Funds Rate a decimales
df['CPI'] = df['CPI'] / 100
df['Fed Funds Rate'] = df['Fed Funds Rate'] / 100

In [ ]:
# Calcular diferencias en puntos básicos
df['dif_CPI_mediana'] = (df['CPI'] - df['CPI_Exp_mediana']) * 10000
df['dif_CPI_promedio'] = (df['CPI'] - df['CPI_Exp_promedio']) * 10000

df['dif_FFR_mediana'] = (df['Fed Funds Rate'] - df['Fed Funds Rate_Exp_mediana']) * 10000
df['dif_FFR_promedio'] = (df['Fed Funds Rate'] - df['Fed Funds Rate_Exp_promedio']) * 10000

# Calcular diferencia
df['dif_NFP_mediana'] = (df['Non farm payrolls'] - df['Non farm payrolls_Exp_mediana'])
df['dif_NFP_promedio'] = (df['Non farm payrolls'] - df['Non farm payrolls_Exp_promedio'])

In [ ]:
# tickers financieros (GICS)
gics_financieras = r'C:\Users\Andy\OneDrive\Desktop\MCD\Tesis\Datos_fuente_Bloomberg\GICS_Finanzas.xlsx'
try:
    df_tickers_financieras = pd.read_excel(gics_financieras)
    columna_ticker_excel = 'Ticker de miembro'
    columna_ticker_df = 'Ticker de miembro'

    if columna_ticker_excel not in df_tickers_financieras.columns:
        raise ValueError(f"La columna '{columna_ticker_excel}' no se encuentra en el archivo Excel.")

    # Seleccionar solo la columna de tickers y renombrarla si es necesario para el merge
    df_tickers_financieras = df_tickers_financieras[[columna_ticker_excel]].rename(columns={columna_ticker_excel: columna_ticker_df})
    # Eliminar duplicados por si acaso
    df_tickers_financieras = df_tickers_financieras.drop_duplicates().reset_index(drop=True)

    print(f"Se cargaron {len(df_tickers_financieras)} tickers del sector financiero.")
    print(df_tickers_financieras.head())

except FileNotFoundError:
    print(f"Error: No se encontró el archivo '{gics_financieras}'")
    # Detener o manejar el error
except ValueError as ve:
     print(f"Error al procesar el Excel: {ve}")
     # Detener o manejar el error
except Exception as e:
    print(f"Ocurrió un error inesperado al leer el Excel: {e}")
    # Detener o manejar el error

In [ ]:
# Lista de columnas numéricas para análisis con mensualización lineal de los ratios 
# y variables macroeconómicas calculadas con mediana
columnas_numericas_lineal_mediana= [
    'P_E',
    'P_B',
    'P_S',
    'P_Share',
    'ROCE_l',
    'EBIT_l',
    'Total Activos_l',
    'Deuda a LP_l',
    'ROA_l',
    'Beneficio neto_l',
    'ROI_l',
    'EV_l',
    'Cap de mercado_l',
    'Deuda a CP_l',
    'Efectivo y equiv_l',
    'dif_CPI_mediana',
    'dif_FFR_mediana',
    'dif_NFP_mediana'
]

In [ ]:
# Lista de columnas numéricas para análisis con mensualización spline de los ratios 
# y variables macroeconómicas calculadas con mediana
columnas_numericas_spline_mediana = [
    'P_E',
    'P_B',
    'P_S',
    'P_Share',
    'ROCE_sp',
    'EBIT_sp',
    'Total Activos_sp',
    'Deuda a LP_sp',
    'ROA_sp',
    'Beneficio neto_sp',
    'ROI_sp',
    'EV_sp',
    'Cap de mercado_sp',
    'Deuda a CP_sp',
    'Efectivo y equiv_sp',
    'dif_CPI_mediana',
    'dif_FFR_mediana',
    'dif_NFP_mediana'
]

In [ ]:
# Lista de columnas numéricas para análisis con mensualización lineal de los ratios 
# y variables macroeconómicas calculadas con promedio
columnas_numericas_lineal_promedio = [
    'P_E',
    'P_B',
    'P_S',
    'P_Share',
    'ROCE_l',
    'EBIT_l',
    'Total Activos_l',
    'Deuda a LP_l',
    'ROA_l',
    'Beneficio neto_l',
    'ROI_l',
    'EV_l',
    'Cap de mercado_l',
    'Deuda a CP_l',
    'Efectivo y equiv_l',
    'dif_CPI_promedio',
    'dif_FFR_promedio',
    'dif_NFP_promedio'
]

In [ ]:
# Lista de columnas numéricas para análisis con mensualización spline de los ratios 
# y variables macroeconómicas calculadas con promedio
columnas_numericas_spline_promedio = [
    'P_E',
    'P_B',
    'P_S',
    'P_Share',
    'ROCE_sp',
    'EBIT_sp',
    'Total Activos_sp',
    'Deuda a LP_sp',
    'ROA_sp',
    'Beneficio neto_sp',
    'ROI_sp',
    'EV_sp',
    'Cap de mercado_sp',
    'Deuda a CP_sp',
    'Efectivo y equiv_sp',
    'dif_CPI_promedio',
    'dif_FFR_promedio',
    'dif_NFP_promedio'
]

In [ ]:
listas_columnas = {
    'lineal_mediana': columnas_numericas_lineal_mediana,
    'spline_mediana': columnas_numericas_spline_mediana,
    'lineal_promedio': columnas_numericas_lineal_promedio,
    'spline_promedio': columnas_numericas_spline_promedio
}

In [ ]:
listas_columnas

In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │ BLOQUE 1: Importaciones y Funciones Auxiliares                  │
# └─────────────────────────────────────────────────────────────────┘
import pandas as pd
import numpy as np

def pct_missing_by_column(df, columns):
    """Devuelve % de NaNs por columna, ordenado descendente."""
    return (df[columns].isna().mean() * 100).sort_values(ascending=False)

def pct_missing_by_group(df, group_col, features):
    """Devuelve Series con el máximo % de NaNs en cualquier feature por grupo."""
    missing = df.groupby(group_col)[features]\
                .apply(lambda g: g.isna().mean()*100)
    return missing.max(axis=1)

def filter_companies_by_target_missing(df, group_col, target_col):
    """Filtra y devuelve df sin empresas que tengan ANY NaN en target_col."""
    ok = ~df.groupby(group_col)[target_col].apply(lambda s: s.isna().any())
    valid_companies = ok[ok].index
    return df[df[group_col].isin(valid_companies)].copy()

def impute_by_group(df, group_col, features, methods=('ffill',), fill_value=0): # <-- Cambiar default
    """Imputa NaNs por grupo usando .ffill() y termina con fillna(fill_value)."""
    df2 = df.sort_values([group_col, 'Fecha']).copy()
    for m in methods:
        df2[features] = df2.groupby(group_col)[features]\
                            .transform(lambda g: getattr(g, m)())
    return df2.fillna(fill_value)

def create_lags(df, group_col, date_col, features, lag=1):
    """Genera columnas de lag para cada feature dentro de cada grupo."""
    df2 = df.sort_values([group_col, date_col]).copy()
    for feat in features:
        df2[f"{feat}_lag{lag}"] = df2.groupby(group_col)[feat].shift(lag)
    return df2


In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │ BLOQUE 2: Filtrar Empresas Financieras
# └─────────────────────────────────────────────────────────────────┘
# Asume: df (DataFrame principal) y df_tickers_financieras ya cargados.
#        La primera columna de df_tickers_financieras contiene los tickers.
#        La columna de tickers/empresas en df se llama 'Empresa'.

columna_ticker_en_df_principal = 'Empresa' # Columna en df para hacer el match
columna_con_tickers_financieros = df_tickers_financieras.columns[0] # Primera columna de df_tickers_financieras

print(f"\nFiltrando DataFrame principal ('df') para tickers financieros...")
print(f"Se usarán los tickers de la columna '{columna_con_tickers_financieros}' del archivo de tickers financieros.")
print(f"Se buscarán coincidencias en la columna '{columna_ticker_en_df_principal}' del DataFrame principal.")

# Verificar que la columna exista en df
if columna_ticker_en_df_principal not in df.columns:
    print(f"ERROR: La columna '{columna_ticker_en_df_principal}' no existe en el DataFrame principal 'df'.")
    print(f"Columnas disponibles en 'df': {df.columns.tolist()}")
    df_fin = pd.DataFrame() # Crear df vacío para evitar errores posteriores y detener
    # Podrías lanzar un error aquí: raise ValueError(f"Columna '{columna_ticker_en_df_principal}' no en df.")
else:
    # Obtener la lista única de tickers financieros cargados
    tickers_financieros_lista_unica = df_tickers_financieras[columna_con_tickers_financieros].unique()
    
    # Filtrar df usando la lista de tickers
    df_fin = df[df[columna_ticker_en_df_principal].isin(tickers_financieros_lista_unica)].copy()

    num_empresas_filtradas_fin = df_fin[columna_ticker_en_df_principal].nunique()
    print(f"\nDataFrame filtrado ('df_fin') contiene:")
    print(f"  - {len(df_fin)} filas")
    print(f"  - {num_empresas_filtradas_fin} empresas financieras únicas.")

    if df_fin.empty:
        print("\n¡ADVERTENCIA! El DataFrame filtrado 'df_fin' está vacío.")
        print("Posibles causas:")
        print(f"  - No hay coincidencias entre los tickers de '{columna_con_tickers_financieros}' y los tickers en '{columna_ticker_en_df_principal}'.")
        print(f"  - Verifica los formatos de los tickers (ej. 'AAPL' vs 'AAPL US Equity').")
    else:
        print("\nPrimeras 5 filas del DataFrame filtrado ('df_fin'):")
        print(df_fin.head())

In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │ BLOQUE 3: Conversión Masiva a Numérico                         │
# └─────────────────────────────────────────────────────────────────┘
# Crear set de features base (sin 'Empresa' ni 'Fecha'), añadir 'P_E' si existe
features_base = {
    col for cols in listas_columnas.values()
            for col in cols if col not in ('Empresa','Fecha')
}
if 'P_E' in df_fin.columns:
    features_base.add('P_E')
numeric_cols = [c for c in sorted(features_base) if c in df_fin.columns]

# Vectorizado
nan_before = df_fin[numeric_cols].isna().sum()
df_fin[numeric_cols] = df_fin[numeric_cols].apply(pd.to_numeric, errors='coerce')
nan_after = df_fin[numeric_cols].isna().sum()
print("NaNs añadidos:", (nan_after - nan_before).sum())

In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │ BLOQUE 4: Filtrar Empresas con NaNs en el Target P_E           │
# └─────────────────────────────────────────────────────────────────┘
df_fin_no_PE_NaN = filter_companies_by_target_missing(df_fin, 'Empresa', 'P_E')
print("Empresas tras filtrar P_E NaN:", df_fin_no_PE_NaN['Empresa'].nunique())

In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │ BLOQUE 4: Excluir Columnas Críticas con Demasiados NaNs        │
# └─────────────────────────────────────────────────────────────────┘
col_pct = pct_missing_by_column(df_fin_no_PE_NaN, numeric_cols)
col_pct

In [ ]:
umbral_col = 5.0   # % máximo de NaNs tolerable
cols_to_exclude = col_pct[col_pct > umbral_col].index.tolist()
features_kept = [c for c in numeric_cols if c not in cols_to_exclude]
print(f"Columnas excluidas (> {umbral_col}% NaNs):", cols_to_exclude)

In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │ BLOQUE 6: Excluir Empresas con Demasiados NaNs en Features     │
# └─────────────────────────────────────────────────────────────────┘
umbral_grp = 10   # % máximo de NaNs por empresa
grp_pct = pct_missing_by_group(df_fin_no_PE_NaN, 'Empresa', features_kept)
empresas_final = grp_pct[grp_pct <= umbral_grp].index.tolist()
df_fin_final = df_fin_no_PE_NaN[df_fin_no_PE_NaN['Empresa'].isin(empresas_final)].copy()
print("Empresas finales (<= {umbral_grp}% NaNs):", len(empresas_final))


In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │ BLOQUE 7: Imputar NaNs Restantes                                │
# └─────────────────────────────────────────────────────────────────┘
df_imputed = impute_by_group(df_fin_final, 'Empresa', features_kept)
print("NaNs totales después de imputar:", df_imputed[features_kept].isna().sum().sum())


In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │ BLOQUE 8: Crear DataFrames con Lag (MODIFICADO para usar features_kept)                │
# └─────────────────────────────────────────────────────────────────┘
dfs_shifted = {}
# Asegúrate que 'features_kept' esté definido y sea la lista de columnas que SÍ quieres usar.
# 'P_E' se maneja por separado como target, no necesita estar en features_kept para esto.

for metodo, cols_originales_del_metodo in listas_columnas.items():
    # 1. Intersectar las columnas originales del método con 'features_kept'
    #    Estas son las features que pasaron el filtro de NaNs Y están definidas para este método.
    feats_para_laggear = [
        col for col in cols_originales_del_metodo
        if col in features_kept  # Solo las que pasaron el filtro de NaNs global
           and col in df_imputed.columns # Y existen en df_imputed
           and col not in ('Empresa', 'Fecha', 'P_E') # Y no son IDs o el target mismo
    ]

    if not feats_para_laggear:
        print(f"Método '{metodo}': No hay features válidas (de 'features_kept') para crear lags. Se omite.")
        continue

    # 2. Preparar el DataFrame base solo con las features validadas + IDs + Target
    #    Columnas necesarias para df_base: 'Empresa', 'Fecha', 'P_E' (target), y las feats_para_laggear
    columnas_para_df_base = ['Empresa', 'Fecha']
    if 'P_E' in df_imputed.columns:
        columnas_para_df_base.append('P_E')
    
    # Añadir las features a laggear, evitando duplicados si 'P_E' estuviera accidentalmente en feats_para_laggear
    columnas_para_df_base.extend(f for f in feats_para_laggear if f not in columnas_para_df_base)
    
    # Asegurar que todas las columnas seleccionadas existan en df_imputed
    columnas_para_df_base = [col for col in columnas_para_df_base if col in df_imputed.columns]
    
    # Verificar que aún tenemos las columnas esenciales
    if not all(essential_col in columnas_para_df_base for essential_col in ['Empresa', 'Fecha', 'P_E']):
        print(f"Método '{metodo}': Faltan columnas esenciales ('Empresa', 'Fecha', 'P_E') después de filtrar. Se omite.")
        continue
        
    df_base = df_imputed[columnas_para_df_base].copy()

    # 3. Generar los lags para las features seleccionadas
    df_lagged = create_lags(df_base, 'Empresa', 'Fecha', feats_para_laggear, lag=1)

    # 4. Eliminar filas donde cualquier lag de las features seleccionadas sea NaN
    #    Esto es importante porque create_lags introduce NaNs en la primera fila de cada grupo.
    nombres_columnas_lag = [f"{feat}_lag1" for feat in feats_para_laggear]
    
    # Comprobar que las columnas de lag existen antes de intentar dropear NaNs en ellas
    nombres_columnas_lag_existentes = [col for col in nombres_columnas_lag if col in df_lagged.columns]
    if not nombres_columnas_lag_existentes:
         print(f"Método '{metodo}': No se generaron columnas de lag. Se omite df_lagged.dropna().")
    else:
        df_lagged.dropna(subset=nombres_columnas_lag_existentes, inplace=True)

    if df_lagged.empty:
        print(f"Método '{metodo}': El DataFrame quedó vacío después de crear lags y dropna. Posiblemente todas las filas tenían NaNs en los lags.")
        # No añadir a dfs_shifted o manejarlo como un método sin datos válidos
    else:
        dfs_shifted[metodo] = df_lagged
        print(f"Método '{metodo}': {df_lagged.shape[0]} filas, {df_lagged.shape[1]} columnas. Features laggeadas: {feats_para_laggear}")


In [ ]:
# Celda: BLOQUE DE EXPERIMENTOS (ENTRENAR SOLO CON LAGS Y GUARDAR PAQUETE COMPLETO)

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import joblib

# --- Modelos ---
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# --- Métricas y Preprocesamiento ---
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, GroupKFold # BaseCrossValidator no se usa directamente
from sklearn.pipeline import Pipeline

# Ignorar warnings comunes (opcional)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
pd.options.mode.chained_assignment = None

# =====================================
# CLASE AUXILIAR DropColumns (Definir una vez)
# =====================================
class DropColumns(BaseEstimator, TransformerMixin):
    """Transformer para eliminar columnas especificadas en un Pipeline."""
    def __init__(self, columns=None):
        self.columns = columns if columns is not None else []
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            return X
        X_transformed = X.copy()
        return X_transformed.drop(columns=self.columns, errors='ignore')

# =====================================
# FUNCIONES AUXILIARES
# =====================================
def rmse(y_true, y_pred):
    y_true_np, y_pred_np = np.asarray(y_true), np.asarray(y_pred)
    mask = np.isfinite(y_true_np) & np.isfinite(y_pred_np)
    if not np.all(mask): y_true_np, y_pred_np = y_true_np[mask], y_pred_np[mask]
    if len(y_true_np) == 0: return np.nan
    return np.sqrt(mean_squared_error(y_true_np, y_pred_np))

rmse_scorer = make_scorer(rmse, greater_is_better=False)

def train_test_split_by_empresa(df_model_input, n_empresas_prueba):
    df_model = df_model_input.copy()
    if 'Empresa' not in df_model.columns: raise KeyError("'Empresa' necesaria")
    if 'P_E' not in df_model.columns: raise KeyError("'P_E' (target) necesaria")

    empresas_unicas = df_model['Empresa'].unique()
    n_total_empresas = len(empresas_unicas)

    if n_total_empresas == 0: return None, None, None, None
    if n_total_empresas == 1:
        X = df_model.drop(columns=['P_E']); y = df_model['P_E']
        return X.copy(), y.copy(), pd.DataFrame(columns=X.columns), pd.Series(dtype=y.dtype)

    # Asegurar que n_empresas_prueba sea válido (entre 0 y n_total_empresas - 1)
    n_empresas_prueba_valido = max(0, min(n_empresas_prueba, n_total_empresas - 1 if n_total_empresas > 0 else 0))
    if n_empresas_prueba_valido != n_empresas_prueba:
        print(f"  Advertencia train_test_split: n_empresas_prueba ajustado de {n_empresas_prueba} a {n_empresas_prueba_valido}")
    n_empresas_prueba = n_empresas_prueba_valido


    if n_empresas_prueba == 0:
        empresas_entrenamiento = empresas_unicas; empresas_prueba = np.array([])
    else:
        empresas_entrenamiento, empresas_prueba = train_test_split(
            empresas_unicas, test_size=n_empresas_prueba, random_state=42, shuffle=True)

    mask_entrenamiento = df_model['Empresa'].isin(empresas_entrenamiento)
    X = df_model.drop(columns=['P_E']); y = df_model['P_E']
    X_train, y_train = X[mask_entrenamiento].copy(), y[mask_entrenamiento].copy()

    if n_empresas_prueba > 0 and len(empresas_prueba) > 0:
        mask_prueba = df_model['Empresa'].isin(empresas_prueba)
        X_test, y_test = X[mask_prueba].copy(), y[mask_prueba].copy()
        # Si X_test es vacío pero se esperaban datos de prueba (mask_prueba no vacía), inicializarlo vacío con columnas correctas
        if X_test.empty and not mask_prueba.empty() and not X_train.empty : 
             X_test = pd.DataFrame(columns=X_train.columns)
             y_test = pd.Series(dtype=y_train.dtype)

    else: # No hay empresas de prueba o X_train está vacío
        X_test_cols = X_train.columns if not X_train.empty else (X.columns if not X.empty else [])
        y_test_dtype = y_train.dtype if not y_train.empty else (y.dtype if not y.empty else float)
        X_test, y_test = pd.DataFrame(columns=X_test_cols), pd.Series(dtype=y_test_dtype)


    if X_train.empty and n_total_empresas > 0 : # Si X_train está vacío pero había datos, es un error
        print("ERROR train_test_split: X_train está vacío después del split.")
        return None, None, None, None
        
    return X_train, y_train, X_test, y_test


def evaluate_model(y_test, y_pred):
    y_test_np, y_pred_np = np.asarray(y_test), np.asarray(y_pred)
    mask = np.isfinite(y_test_np) & np.isfinite(y_pred_np)
    if not np.all(mask): y_test_np, y_pred_np = y_test_np[mask], y_pred_np[mask]
    if len(y_test_np) == 0: return {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}
    
    mae = mean_absolute_error(y_test_np, y_pred_np)
    rmse_val = rmse(y_test_np, y_pred_np) # rmse ya maneja NaNs/Infs
    r2 = np.nan
    if len(y_test_np) > 1 and np.var(y_test_np) > 1e-9:
        try: r2 = r2_score(y_test_np, y_pred_np)
        except ValueError: pass
    elif len(y_test_np) > 0 and mean_squared_error(y_test_np, y_pred_np) < 1e-9: r2 = 1.0
    elif len(y_test_np) > 0: r2 = 0.0
    return {'RMSE': rmse_val, 'MAE': mae, 'R2': r2}

def sort_by_timestep(X_df, y_series):
    X, y = X_df.copy(), y_series.copy()
    if 'time_step' not in X.columns: return X, y
    if not isinstance(y, pd.Series) or not X.index.equals(y.index): # Asegurar que los índices coincidan
        if len(y) == len(X): y = pd.Series(np.asarray(y), index=X.index, name=getattr(y, 'name', 'target'))
        else: print("Error sort_by_timestep: y no alineable con X."); return X, y
    X_sorted = X.sort_values('time_step'); y_sorted = y.loc[X_sorted.index]
    return X_sorted, y_sorted

# ================================================================================
# BLOQUE DE EXPERIMENTOS (ENTRENAR SOLO CON CARACTERÍSTICAS REZAGADAS '_lag1')
# ================================================================================

# ASUNCIONES ANTES DE CORRER ESTE BLOQUE:
# 1. `dfs_shifted`: Diccionario. Key=`metodo`. Value (`df_lag1_original_con_todo`) es un DataFrame que contiene:
#      - IDs: 'Empresa', 'Fecha'.
#      - Target: 'P_E'.
#      - COLUMNAS BASE ORIGINALES del método.
#      - COLUMNAS REZAGADAS `_lag1` (el número de estas depende de filtros previos como `features_kept` en BLOQUE 8).
#    El número de columnas `_lag1` en `dfs_shifted[metodo]` determinará las features para el modelo de ESE `metodo`.

results = []
start_time_total = time.time()
use_log_transform = True
test_size_ratio = 0.20

for metodo, df_lag1_original_con_todo in dfs_shifted.items():
    start_time_config = time.time()
    print("\n" + "="*80)
    print(f"Procesando Método: {metodo} {'(CON Log Transform)' if use_log_transform else '(SIN Log Transform)'} (SOLO LAGS COMO PREDICTORES)")
    print("="*80 + "\n")

    if df_lag1_original_con_todo.empty:
        print(f"  ⇨ ADVERTENCIA: df_lag1_original_con_todo para el método {metodo} está vacío. Omitiendo.")
        results.append({'Modelo': 'N/A', 'Validacion_Temp': False, 'Busqueda_HP': False, 'Config_Key': metodo, 'Metodo': metodo,
                        'Lag_Config': 'lag1_solo_pred', 'Log_Transform': use_log_transform, 'RMSE': np.nan, 'MAE': np.nan,
                        'R2': np.nan, 'Best_Params': None, 'pipeline': None, 'columnas_base_usadas': [], 'n_features_modelo': 0, 'Error': 'Input DataFrame for method is empty'})
        continue

    df_temp_for_method = df_lag1_original_con_todo.copy()

    if 'time_step' not in df_temp_for_method.columns:
        df_temp_for_method.sort_values(['Empresa', 'Fecha'], inplace=True)
        df_temp_for_method['time_step'] = df_temp_for_method.groupby('Empresa').cumcount()

    predictor_cols_solo_lag1 = sorted([c for c in df_temp_for_method.columns if c.endswith('_lag1')])
    
    if not predictor_cols_solo_lag1:
        print(f"  ⇨ ADVERTENCIA: No se encontraron columnas _lag1 en dfs_shifted['{metodo}']. Omitiendo método {metodo}.")
        results.append({'Modelo': 'N/A', 'Validacion_Temp': False, 'Busqueda_HP': False, 'Config_Key': metodo, 'Metodo': metodo,
                        'Lag_Config': 'lag1_solo_pred', 'Log_Transform': use_log_transform, 'RMSE': np.nan, 'MAE': np.nan,
                        'R2': np.nan, 'Best_Params': None, 'pipeline': None, 'columnas_base_usadas': [], 'n_features_modelo': 0, 'Error': f'No lag features found in dfs_shifted for {metodo}'})
        continue
    
    columnas_base_que_generaron_estos_lags = sorted(list(set(lag_col.replace('_lag1', '') for lag_col in predictor_cols_solo_lag1)))
    print(f"  ⇨ Método {metodo}: Se usarán {len(predictor_cols_solo_lag1)} predictores (_lag1).")
    print(f"    Estos lags fueron generados a partir de {len(columnas_base_que_generaron_estos_lags)} columnas base: {columnas_base_que_generaron_estos_lags[:5]}...")

    cols_for_df_model = ['Empresa', 'Fecha', 'time_step', 'P_E'] + predictor_cols_solo_lag1
    final_cols_for_df_model = sorted(list(set(c for c in cols_for_df_model if c in df_temp_for_method.columns))) # Unicas y existentes
    
    essential_check = ['Empresa', 'Fecha', 'time_step', 'P_E']
    if not all(ec in final_cols_for_df_model for ec in essential_check):
        missing_ess_cols = [ec for ec in essential_check if ec not in final_cols_for_df_model]
        print(f"  ⇨ ERROR: Faltan columnas esenciales ({missing_ess_cols}) para construir df_model para {metodo}. Omitiendo.")
        results.append({'Modelo': 'N/A', 'Validacion_Temp': False, 'Busqueda_HP': False, 'Config_Key': metodo, 'Metodo': metodo,
                        'Lag_Config': 'lag1_solo_pred', 'Log_Transform': use_log_transform, 'RMSE': np.nan, 'MAE': np.nan,
                        'R2': np.nan, 'Best_Params': None, 'pipeline': None, 'columnas_base_usadas': columnas_base_que_generaron_estos_lags, 'n_features_modelo': len(predictor_cols_solo_lag1), 'Error': f'Missing essential cols for df_model: {missing_ess_cols}'})
        continue
        
    df_model = df_temp_for_method[final_cols_for_df_model].copy()

    n_total_empresas_metodo = df_model['Empresa'].nunique()
    n_empresas_test_metodo = 0
    if n_total_empresas_metodo > 1:
        n_empresas_test_metodo = max(1, int(round(n_total_empresas_metodo * test_size_ratio)))
        if n_total_empresas_metodo - n_empresas_test_metodo < 1 : n_empresas_test_metodo = n_total_empresas_metodo - 1
    
    X_train, y_train, X_test, y_test = train_test_split_by_empresa(df_model, n_empresas_test_metodo)
    
    if X_train is None or X_train.empty:
        print(f"  Split train/test fallido o X_train vacío para {metodo}. Omitiendo método.")
        results.append({'Modelo': 'N/A', 'Validacion_Temp': False, 'Busqueda_HP': False, 'Config_Key': metodo, 'Metodo': metodo,
                        'Lag_Config': 'lag1_solo_pred', 'Log_Transform': use_log_transform, 'RMSE': np.nan, 'MAE': np.nan,
                        'R2': np.nan, 'Best_Params': None, 'pipeline': None, 'columnas_base_usadas': columnas_base_que_generaron_estos_lags, 'n_features_modelo': len(predictor_cols_solo_lag1), 'Error': 'Train/Test split failed or X_train empty'})
        continue
    
    print(f"  ⇨ Shapes post-split: X_train: {X_train.shape}, y_train: {y_train.shape}, X_test: {X_test.shape if X_test is not None else 'None'}, y_test: {y_test.shape if y_test is not None else 'None'}")

    y_train_transformed = y_train.copy()
    if use_log_transform:
        y_numeric = pd.to_numeric(y_train, errors='coerce')
        median_y = y_numeric.median() if not y_numeric.isnull().all() else 0.0
        y_numeric = y_numeric.fillna(median_y) # Rellenar NaNs antes de chequear negativos
        y_numeric.loc[y_numeric < 0] = 0 # Asignar 0 a negativos para log1p
        y_train_transformed = np.log1p(y_numeric)
        median_yt = y_train_transformed.median() if not y_train_transformed.isnull().all() else 0.0
        y_train_transformed = y_train_transformed.fillna(median_yt)


    X_train_s, y_train_s = sort_by_timestep(X_train, y_train_transformed)
    X_test_s, y_test_s_original = pd.DataFrame(columns=X_train.columns), pd.Series(dtype=y_train.dtype) # Inicializar
    if X_test is not None and not X_test.empty:
        X_test_s, y_test_s_original = sort_by_timestep(X_test, y_test)

    param_grid_rf   = {'model__max_depth':[10, 20, None],'model__min_samples_split':[5, 10],'model__n_estimators':[100, 200]}
    param_grid_xgb  = {'model__n_estimators':[100, 200],'model__max_depth':[3, 5, 7],'model__learning_rate':[0.1, 0.05]}
    param_grid_lgbm = {'model__n_estimators':[100, 200],'model__learning_rate':[0.1, 0.05],'model__max_depth':[3, 5, 7],'model__num_leaves':[15, 31]}
    param_grid_cb   = {'model__iterations':[100, 200],'model__learning_rate':[0.1, 0.05],'model__depth':[3, 5, 7],'model__l2_leaf_reg':[1, 3]}

    scenarios = {
        'RF_Simple':  {'model': RandomForestRegressor(random_state=42, n_jobs=-1), 'hp': False, 'vt': False},
        'RF_HP':      {'model': RandomForestRegressor(random_state=42, n_jobs=-1), 'hp': True,  'vt': False, 'grid': param_grid_rf},
        'XGB_Simple': {'model': XGBRegressor(random_state=42, n_jobs=-1), 'hp': False, 'vt': False},
        'XGB_HP':     {'model': XGBRegressor(random_state=42, n_jobs=-1), 'hp': True,  'vt': False, 'grid': param_grid_xgb},
        'LGBM_Simple':{'model': LGBMRegressor(random_state=42, n_jobs=-1, verbosity=-1), 'hp': False, 'vt': False},
        'LGBM_HP':    {'model': LGBMRegressor(random_state=42, n_jobs=-1, verbosity=-1), 'hp': True,  'vt': False, 'grid': param_grid_lgbm},
        'CB_Simple':  {'model': CatBoostRegressor(random_state=42, verbose=0, allow_writing_files=False), 'hp': False, 'vt': False},
        'CB_HP':      {'model': CatBoostRegressor(random_state=42, verbose=0, allow_writing_files=False), 'hp': True,  'vt': False, 'grid': param_grid_cb},
    }

    for escenario_nombre, escenario_cfg in scenarios.items():
        tiempo_esc_inicio = time.time()
        print(f"\n  --- Escenario: {escenario_nombre} para Método: {metodo} ---")
        current_base_model, perform_hp_search, use_temporal_cv = escenario_cfg['model'], escenario_cfg['hp'], escenario_cfg['vt']
        X_fit_data, y_fit_data = (X_train_s, y_train_s) if use_temporal_cv else (X_train, y_train_transformed)
        X_eval_data, y_eval_data_original = (X_test_s, y_test_s_original) if use_temporal_cv else (X_test, y_test)

        pipeline_obj = Pipeline([('drop_ids', DropColumns(columns=['Empresa', 'Fecha', 'time_step'])), ('model', current_base_model)])
        final_fitted_model, hp_best_params, error_msg_escenario = None, None, None

        try:
            if perform_hp_search:
                cv_strategy, cv_fit_params = None, {}
                n_groups_cv = X_fit_data['Empresa'].nunique() if 'Empresa' in X_fit_data.columns else 0

                can_do_cv = True
                if use_temporal_cv and n_groups_cv >= 2:
                    X_cv_input = X_fit_data.sort_values(['Empresa', 'time_step']); y_cv_input = y_fit_data.loc[X_cv_input.index]
                    groups_cv = X_cv_input['Empresa']
                    n_splits = min(4, n_groups_cv); n_splits = max(2, n_splits) # GroupKFold min 2
                    if len(X_cv_input) < n_splits: can_do_cv = False; print(f"    Pocas muestras ({len(X_cv_input)}) para {n_splits} splits en GroupKFold.")
                    else: cv_strategy = GroupKFold(n_splits=n_splits); cv_fit_params = {'groups': groups_cv}; print(f"    Usando GroupKFold CV con {n_splits} splits.")
                elif len(X_fit_data) >= 2 : # KFold estándar
                    kfold_splits = min(3, len(X_fit_data)); kfold_splits = max(2, kfold_splits) # KFold min 2
                    if len(X_fit_data) < kfold_splits : can_do_cv = False; print(f"    Pocas muestras ({len(X_fit_data)}) para {kfold_splits} splits en KFold.")
                    else: cv_strategy = KFold(n_splits=kfold_splits, shuffle=True, random_state=42); print(f"    Usando KFold CV con {kfold_splits} splits.")
                else: can_do_cv = False; print(f"    No hay suficientes muestras ({len(X_fit_data)}) para CV. Omitiendo HP search.")
                
                if can_do_cv:
                    gs = GridSearchCV(pipeline_obj, escenario_cfg['grid'], scoring=rmse_scorer, cv=cv_strategy, n_jobs=-1, refit=True, error_score='raise', verbose=0)
                    gs.fit(X_fit_data, y_fit_data, **cv_fit_params)
                    final_fitted_model, hp_best_params = gs.best_estimator_, gs.best_params_
                    print(f"    Mejores parámetros: {hp_best_params}")
                else: perform_hp_search = False # Forzar no HP search y fit simple
            
            if not perform_hp_search or not final_fitted_model: # Si HP search se omitió o falló, fit simple
                pipeline_obj.fit(X_fit_data, y_fit_data)
                final_fitted_model = pipeline_obj
        except Exception as e: error_msg_escenario = str(e); print(f"    ERROR entrenamiento/HP search {escenario_nombre}: {e}")

        eval_metrics, model_n_features = {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}, "N/A"
        if final_fitted_model:
            try:
                model_estimator_step = final_fitted_model.steps[-1][1]
                model_n_features = getattr(model_estimator_step, 'n_features_in_', 'N/A')
                print(f"    >> Estimador final '{type(model_estimator_step).__name__}' vio {model_n_features} features.")
                if isinstance(model_n_features, int) and model_n_features != len(predictor_cols_solo_lag1):
                     print(f"       ¡ADVERTENCIA! n_features ({model_n_features}) no coincide con num. de lags ({len(predictor_cols_solo_lag1)}).")

                if X_eval_data is not None and not X_eval_data.empty and y_eval_data_original is not None and not y_eval_data_original.empty:
                    pred_transformed = final_fitted_model.predict(X_eval_data)
                    pred_original_scale = np.expm1(pred_transformed) if use_log_transform else pred_transformed
                    if use_log_transform: pred_original_scale[~np.isfinite(pred_original_scale)] = np.nan
                    eval_metrics = evaluate_model(y_eval_data_original, pred_original_scale)
                    print(f"    Métricas Test {escenario_nombre}: RMSE={eval_metrics['RMSE']:.4f}, MAE={eval_metrics['MAE']:.4f}, R2={eval_metrics['R2']:.4f}")
                else: print("    No hay datos de X_eval/y_eval para evaluar.")
            except Exception as e_eval: error_msg_escenario = (error_msg_escenario + f" | EvalError: {str(e_eval)}") if error_msg_escenario else f"EvalError: {str(e_eval)}"; print(f"    ERROR evaluación {escenario_nombre}: {e_eval}")
        
        results.append({
            'Modelo': escenario_nombre.split('_')[0], 'Validacion_Temp': use_temporal_cv,
            'Busqueda_HP': perform_hp_search and escenario_cfg['hp'], 'Config_Key': metodo, 'Metodo': metodo,
            'Lag_Config': 'lag1_solo_pred', 'Log_Transform': use_log_transform, 'RMSE': eval_metrics['RMSE'],
            'MAE': eval_metrics['MAE'], 'R2': eval_metrics['R2'], 'Best_Params': hp_best_params,
            'pipeline': final_fitted_model, 'columnas_base_usadas': columnas_base_que_generaron_estos_lags,
            'n_features_modelo': model_n_features, 'Error': error_msg_escenario})
        print(f"    Tiempo escenario {escenario_nombre}: {time.time() - tiempo_esc_inicio:.2f}s")
    print(f"\nTiempo total Método {metodo}: {time.time() - start_time_config:.2f}s")
print(f"\n\nTiempo total ejecución experimentos: {(time.time() - start_time_total)/60:.2f} minutos")

# ================================================================================
# CREAR TABLA DE RESULTADOS Y GUARDAR MEJOR MODELO (PAQUETE COMPLETO)
# ================================================================================
if results:
    df_results_all_models = pd.DataFrame(results)
    print("\n" + "="*80 + "\n--- Resultados Finales (Modelos Entrenados SOLO CON LAGS) ---\n" + f"Total resultados: {len(df_results_all_models)}\n")
    cols_display_final = ['Metodo', 'Modelo', 'Busqueda_HP', 'RMSE', 'MAE', 'R2', 'n_features_modelo', 'columnas_base_usadas', 'Best_Params', 'Error']
    cols_to_show_final = [col for col in cols_display_final if col in df_results_all_models.columns]
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 2000, 'display.float_format', '{:.4f}'.format):
        print(df_results_all_models.sort_values(by='RMSE', ascending=True, na_position='last')[cols_to_show_final].to_string(index=False))

    valid_results_for_best = df_results_all_models[df_results_all_models['RMSE'].notna() & df_results_all_models['pipeline'].notna() & df_results_all_models['Error'].isna()].copy()
    if not valid_results_for_best.empty:
        best_model_idx = valid_results_for_best['RMSE'].idxmin()
        best_model_full_info = valid_results_for_best.loc[best_model_idx]
        
        model_package_to_save = {
            'pipeline': best_model_full_info['pipeline'],
            'metodo_features': best_model_full_info['Metodo'], # Clave corregida
            'log_transform_target': best_model_full_info['Log_Transform'],
            'columnas_base_para_lags': best_model_full_info['columnas_base_usadas'],
            'n_features_in_estimator': best_model_full_info['n_features_modelo']}
        
        filename_best_model_package = f"best_model_package_{best_model_full_info['Metodo']}.joblib"
        joblib.dump(model_package_to_save, filename_best_model_package)
        
        print("\n" + "="*80 + "\nPAQUETE DEL MEJOR MODELO (SOLO LAGS) SELECCIONADO Y GUARDADO:\n" +
              f"  Archivo: {filename_best_model_package}\n" +
              f"  Método de Features Original: {best_model_full_info['Metodo']}\n" +
              f"  Columnas Base (de los lags usados): {len(best_model_full_info['columnas_base_usadas'])} {str(best_model_full_info['columnas_base_usadas'][:10]) + ('...' if len(best_model_full_info['columnas_base_usadas']) > 10 else '')}\n" +
              f"  Num. Features para Estimador (lags): {best_model_full_info['n_features_modelo']}\n" +
              f"  Transformación Log Target: {best_model_full_info['Log_Transform']}\n" +
              f"  RMSE en Test: {best_model_full_info['RMSE']:.4f}\n" +
              f"  MAE en Test: {best_model_full_info['MAE']:.4f}\n" +
              f"  R2 en Test: {best_model_full_info['R2']:.4f}\n" +
              f"  Hiperparámetros: {best_model_full_info['Best_Params']}\n" +
              f"  Pipeline: {best_model_full_info['pipeline']}\n" + "="*80)
    else: print("\nNo se encontraron resultados válidos para seleccionar y guardar el mejor modelo.")
else: print("\nNo se generaron resultados en los experimentos.")

In [ ]:
print("\nTabla de Resultados:")
df_results_all_models

In [ ]:
# Elección entre modelos

def select_candidate_models(df, n=3):
    """
    Selecciona los n modelos candidatos basándose únicamente en el RMSE (de menor a mayor).
    """
    return df.sort_values('RMSE').head(n)

In [ ]:
# Candidatos a mejor modelo
top_candidates = select_candidate_models(df_results_all_models, n=3)
print("Top candidatos según RMSE")
top_candidates

In [ ]:
# Seleccionar el mejor pipeline y guardarlo
if not df_results_all_models.empty:
    best_idx = df_results_all_models['RMSE'].idxmin()
    best_pipeline_info = df_results_all_models.loc[best_idx]

    # --- ESTA LÍNEA ES IMPORTANTE ---
    # Asegúrate de que 'pipeline' sea la columna correcta en df_results_all_models
    # que contiene el objeto Pipeline entrenado.
    best_pipeline = best_pipeline_info['pipeline'] 
    # --- ----------------------- ---

    metodo_best_model = best_pipeline_info['Metodo'] # o 'Config_Key', depende de cómo lo llamaste
    log_transform_best_model = best_pipeline_info['Log_Transform']
    
    # Nombre del archivo (quizás quieras incluir el método en el nombre para claridad)
    nombre_archivo_modelo_guardado = f"best_pipeline_model_{metodo_best_model}.pkl"
    # O simplemente:
    # nombre_archivo_modelo_guardado = "best_pipeline_model.pkl"


    joblib.dump(best_pipeline, nombre_archivo_modelo_guardado)
    
    print(f"Modelo ganador (Método: {metodo_best_model}, LogTransform: {log_transform_best_model}) guardado en {nombre_archivo_modelo_guardado}")
    print(f"Pipeline guardado: {best_pipeline}")

else:
    print("No hay resultados de modelos para seleccionar el mejor.")
    # Aquí deberías manejar el caso de que no se haya podido entrenar ningún modelo
    metodo_best_model = None 
    log_transform_best_model = None
    best_pipeline = None
    # Y consecuentemente, la parte de DLocal no podría ejecutarse o necesitaría un modelo por defecto.

## Dlocal

In [ ]:
# Cargar los datos desde el archivo Excel
file_path = 'C:/Users/Andy/OneDrive/Desktop/MCD/Tesis/Datos_fuente_Bloomberg/en valores/serie completa 2014-2024/Dataset/dataset_dlocal_casi sin vacías.xlsx'
df_dlocal = pd.read_excel(file_path, sheet_name='dataset')

In [ ]:
df_dlocal.info()

In [ ]:
# Quitar "k" de las columnas "Non farm payrolls_Exp_mediana" y "Non farm payrolls_Exp_promedio"
df_dlocal['Non farm payrolls_Exp_mediana'] = df_dlocal['Non farm payrolls_Exp_mediana'].astype(str).str.replace('k', '')
df_dlocal['Non farm payrolls_Exp_promedio'] = df_dlocal['Non farm payrolls_Exp_promedio'].astype(str).str.replace('k', '')

# Convertir las columnas a valores numéricos
df_dlocal['Non farm payrolls_Exp_mediana'] = pd.to_numeric(df_dlocal['Non farm payrolls_Exp_mediana'], errors='coerce')
df_dlocal['Non farm payrolls_Exp_promedio'] = pd.to_numeric(df_dlocal['Non farm payrolls_Exp_promedio'], errors='coerce')

df_dlocal['P_B'] = pd.to_numeric(df_dlocal['P_B'], errors='coerce')
#df_dlocal['P_TB'] = pd.to_numeric(df_dlocal['P_TB'], errors='coerce')
df_dlocal['P_S'] = pd.to_numeric(df_dlocal['P_S'], errors='coerce')
#df_dlocal['P_FCF'] = pd.to_numeric(df_dlocal['P_FCF'], errors='coerce')
df_dlocal['P_Share'] = pd.to_numeric(df_dlocal['P_Share'], errors='coerce')

In [ ]:
# Convertir la columna de fecha a formato datetime
df_dlocal['Fecha'] = pd.to_datetime(df_dlocal['Fecha'], format='%Y%m%d')

# Extraer características temporales relevantes
df_dlocal['mes'] = df_dlocal['Fecha'].dt.month
df_dlocal['año'] = df_dlocal['Fecha'].dt.year
df_dlocal['periodo'] = df_dlocal['Fecha'].dt.to_period('M')

In [ ]:
# Convertir CPI y Fed Funds Rate a decimales
df_dlocal['CPI'] = df_dlocal['CPI'] / 100
df_dlocal['Fed Funds Rate'] = df_dlocal['Fed Funds Rate'] / 100

In [ ]:
# Calcular diferencias en puntos básicos
df_dlocal['dif_CPI_mediana'] = (df_dlocal['CPI'] - df_dlocal['CPI_Exp_mediana']) * 10000
df_dlocal['dif_CPI_promedio'] = (df_dlocal['CPI'] - df_dlocal['CPI_Exp_promedio']) * 10000

df_dlocal['dif_FFR_mediana'] = (df_dlocal['Fed Funds Rate'] - df_dlocal['Fed Funds Rate_Exp_mediana']) * 10000
df_dlocal['dif_FFR_promedio'] = (df_dlocal['Fed Funds Rate'] - df_dlocal['Fed Funds Rate_Exp_promedio']) * 10000

# Calcular diferencia
df_dlocal['dif_NFP_mediana'] = (df_dlocal['Non farm payrolls'] - df_dlocal['Non farm payrolls_Exp_mediana'])
df_dlocal['dif_NFP_promedio'] = (df_dlocal['Non farm payrolls'] - df_dlocal['Non farm payrolls_Exp_promedio'])

In [ ]:
import joblib
import os

# 1. Verificar el directorio de trabajo actual
current_dir = os.getcwd()
print(f"Directorio de trabajo actual: {current_dir}")

# 2. Listar archivos en el directorio actual para confirmar visualmente
print("\nArchivos en el directorio actual (primeros 20):")
archivos_en_directorio = os.listdir(current_dir)
for i, item in enumerate(archivos_en_directorio):
    if i < 20: # Imprimir solo los primeros 20 para no saturar
        print(item)
    if item == "best_model_package_spline_promedio.joblib": # Nombre exacto
        print(f"  ---> ¡¡¡ARCHIVO ENCONTRADO EN LA LISTA!!!: {item}")

# 3. Definir el nombre exacto del archivo
nombre_archivo_a_cargar = "best_model_package_spline_promedio.joblib"
ruta_completa_archivo = os.path.join(current_dir, nombre_archivo_a_cargar)
print(f"\nIntentando cargar desde ruta completa: {ruta_completa_archivo}")

# 4. Verificar si el archivo existe usando os.path.exists ANTES de intentar cargarlo
if os.path.exists(ruta_completa_archivo):
    print(f"os.path.exists CONFIRMA que el archivo '{ruta_completa_archivo}' existe.")
    try:
        # 5. Intentar cargar el archivo
        model_package_test = joblib.load(ruta_completa_archivo) # Usar la ruta completa
        print("\n¡Archivo cargado exitosamente con joblib.load!")
        
        # 6. Verificar si es un diccionario y qué claves tiene
        if isinstance(model_package_test, dict):
            print("El archivo cargado ES un diccionario.")
            print(f"Claves en el diccionario: {list(model_package_test.keys())}")
            # Intentar acceder a una clave esperada
            if 'pipeline' in model_package_test:
                print("La clave 'pipeline' está presente.")
            else:
                print("ADVERTENCIA: La clave 'pipeline' NO está presente en el diccionario.")
        else:
            print(f"ADVERTENCIA: El archivo cargado NO es un diccionario. Es de tipo: {type(model_package_test)}")

    except Exception as e:
        print(f"\nERROR al cargar el archivo con joblib.load: {e}")
        print("Esto podría indicar un problema con el archivo en sí (corrupto) o con la versión de joblib/sklearn.")
else:
    print(f"ERROR: os.path.exists indica que el archivo '{ruta_completa_archivo}' NO existe en esa ruta.")
    print("Revisa el nombre del archivo y la ruta.")

In [ ]:
# Celda: BLOQUE DE PREDICCIÓN PARA DLOCAL (Cargando el PAQUETE del Modelo)

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import joblib 

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None): self.columns = columns if columns is not None else []
    def fit(self, X, y=None): return self
    def transform(self, X):
        if not isinstance(X, pd.DataFrame): return X
        return X.copy().drop(columns=self.columns, errors='ignore')

def rmse(y_true, y_pred):
    y_true_np, y_pred_np = np.asarray(y_true), np.asarray(y_pred)
    mask = np.isfinite(y_true_np) & np.isfinite(y_pred_np)
    if not np.all(mask): y_true_np, y_pred_np = y_true_np[mask], y_pred_np[mask]
    if len(y_true_np) == 0: return np.nan
    return np.sqrt(mean_squared_error(y_true_np, y_pred_np))

def evaluate_model(y_test, y_pred):
    y_test_np, y_pred_np = np.asarray(y_test), np.asarray(y_pred)
    mask = np.isfinite(y_test_np) & np.isfinite(y_pred_np)
    if not np.all(mask): y_test_np, y_pred_np = y_test_np[mask], y_pred_np[mask]
    if len(y_test_np) == 0: return {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}
    mae = mean_absolute_error(y_test_np, y_pred_np)
    rmse_val = rmse(y_test_np, y_pred_np)
    r2 = np.nan
    if len(y_test_np) > 1 and np.var(y_test_np) > 1e-9:
        try: r2 = r2_score(y_test_np, y_pred_np)
        except ValueError: pass
    elif len(y_test_np) > 0 and mean_squared_error(y_test_np, y_pred_np) < 1e-9: r2 = 1.0
    elif len(y_test_np) > 0: r2 = 0.0
    return {'RMSE': rmse_val, 'MAE': mae, 'R2': r2}

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
pd.options.mode.chained_assignment = None

# =====================================================================================
# BLOQUE DE PREDICCIÓN PARA DLOCAL (Carga PAQUETE del Modelo y Guarda DF con Lags para Inspección)
# =====================================================================================

# --- 0. CONFIGURACIÓN Y CARGA DEL PAQUETE DEL MODELO ---
# ASUNCIONES: df_dlocal está cargado y con preprocesamiento inicial.
#             El archivo .joblib contiene el DICCIONARIO (paquete del modelo).

# --- ¡¡¡IMPORTANTE!!! ESTABLECE EL NOMBRE CORRECTO DEL ARCHIVO DEL PAQUETE DEL MODELO AQUÍ ---
nombre_paquete_modelo = "best_model_package_spline_promedio.joblib" # <--- CORREGIDO y ASUMIENDO ES EL PAQUETE

print(f"Cargando paquete del modelo desde: {nombre_paquete_modelo}")
try:
    model_package = joblib.load(nombre_paquete_modelo) # Carga el diccionario
    loaded_pipeline = model_package['pipeline']
    metodo_features_original_modelo = model_package['metodo_features'] # Clave correcta para el paquete
    use_log_transform_on_target_modelo = model_package['log_transform_target']
    columnas_base_para_lags_modelo = model_package['columnas_base_para_lags']
    n_features_esperadas_por_estimador_modelo = model_package['n_features_in_estimator']
    
    print("Paquete del modelo cargado exitosamente.")
    print(f"  Pipeline: {loaded_pipeline}")
    print(f"  Método de features original del modelo: {metodo_features_original_modelo}")
    print(f"  Modelo usó Log Transform en Target: {use_log_transform_on_target_modelo}")
    print(f"  COLUMNAS BASE (usadas para generar lags en entrenamiento): {len(columnas_base_para_lags_modelo)} {columnas_base_para_lags_modelo[:5]}...")
    print(f"  N Features esperadas por el estimador final (lags): {n_features_esperadas_por_estimador_modelo}")

except FileNotFoundError:
    raise FileNotFoundError(f"ERROR: No se encontró el archivo del paquete del modelo '{nombre_paquete_modelo}'. Asegúrate de que el nombre y la extensión sean correctos y el archivo exista en el directorio de trabajo actual.")
except KeyError as e_key:
    print(f"ERROR: Clave '{e_key}' no encontrada en el paquete del modelo cargado.")
    print(f"       Esto sugiere que el archivo '{nombre_paquete_modelo}' podría no ser el 'paquete del modelo' (diccionario) esperado, o que la clave es incorrecta.")
    print(f"       Claves disponibles en el objeto cargado (si es un diccionario): {list(model_package.keys()) if 'model_package' in locals() and isinstance(model_package, dict) else 'El objeto cargado no es un diccionario o no se pudo cargar.'}")
    raise RuntimeError(f"Error al cargar el paquete del modelo debido a una clave faltante: {e_key}")
except Exception as e_load:
    raise RuntimeError(f"Error general al cargar el paquete del modelo: {e_load}")

if 'df_dlocal' not in locals() or not isinstance(df_dlocal, pd.DataFrame) or df_dlocal.empty:
     raise NameError("El DataFrame 'df_dlocal' no está definido, está vacío o no es un DataFrame.")

print("\n" + "="*80)
print(f"Preparando datos de DLocal para predicción usando la configuración del modelo '{metodo_features_original_modelo}'.")
print(f"  (Se usarán {len(columnas_base_para_lags_modelo)} columnas base específicas para generar lags)")
print("="*80)

# --- 1. PREPARAR df_dlocal PARA GENERAR LAGS ---
df_dlocal_procesado = df_dlocal.copy()

cols_a_convertir_dlocal = [col for col in columnas_base_para_lags_modelo if col in df_dlocal_procesado.columns]
print(f"Forzando {len(cols_a_convertir_dlocal)} columnas base a numérico en DLocal...")
for col in cols_a_convertir_dlocal:
    if col in df_dlocal_procesado.columns and not pd.api.types.is_numeric_dtype(df_dlocal_procesado[col]):
        df_dlocal_procesado[col] = pd.to_numeric(df_dlocal_procesado[col], errors='coerce')

print("Imputando NaNs pre-lag (ffill + 0) en las columnas base requeridas...")
cols_imputar_prelag_dlocal = [col for col in columnas_base_para_lags_modelo if col in df_dlocal_procesado.columns]
if not cols_imputar_prelag_dlocal:
    print("Advertencia: No hay columnas base para imputar en DLocal.")
else:
    df_dlocal_procesado.sort_values(['Empresa', 'Fecha'], inplace=True)
    if df_dlocal_procesado['Empresa'].nunique() > 1:
        df_dlocal_procesado[cols_imputar_prelag_dlocal] = df_dlocal_procesado.groupby('Empresa')[cols_imputar_prelag_dlocal].ffill()
    else:
        df_dlocal_procesado[cols_imputar_prelag_dlocal] = df_dlocal_procesado[cols_imputar_prelag_dlocal].ffill()
    nans_post_ffill_dlocal = df_dlocal_procesado[cols_imputar_prelag_dlocal].isnull().sum().sum()
    if nans_post_ffill_dlocal > 0:
        print(f"  {nans_post_ffill_dlocal} NaNs restantes en columnas base después de ffill. Rellenando con 0.")
        df_dlocal_procesado[cols_imputar_prelag_dlocal] = df_dlocal_procesado[cols_imputar_prelag_dlocal].fillna(0)
    else: print("  No quedan NaNs en columnas base después de ffill.")

df_dlocal_con_lags_generados_temp = df_dlocal_procesado.copy()
predictor_cols_lag1_para_dlocal_lista_temp = []
columnas_base_efectivas_para_dlocal_lags_temp = []
for cb_modelo_actual_temp in columnas_base_para_lags_modelo:
    if cb_modelo_actual_temp in df_dlocal_con_lags_generados_temp.columns:
        columnas_base_efectivas_para_dlocal_lags_temp.append(cb_modelo_actual_temp)
    else: print(f"  ¡ADVERTENCIA! Columna base '{cb_modelo_actual_temp}' (requerida por el modelo) NO existe en DLocal. No se generará lag para esta.")
if len(columnas_base_efectivas_para_dlocal_lags_temp) != len(columnas_base_para_lags_modelo):
    print(f"  Se esperaban {len(columnas_base_para_lags_modelo)} cols base del modelo, se encontraron {len(columnas_base_efectivas_para_dlocal_lags_temp)} en DLocal para generar lags.")

print(f"Generando {len(columnas_base_efectivas_para_dlocal_lags_temp)} columnas _lag1 para DLocal...")
for col_base_actual_temp in columnas_base_efectivas_para_dlocal_lags_temp:
    lag_col_name_actual_temp = f"{col_base_actual_temp}_lag1"
    df_dlocal_con_lags_generados_temp[lag_col_name_actual_temp] = df_dlocal_con_lags_generados_temp.groupby('Empresa')[col_base_actual_temp].shift(1)
    predictor_cols_lag1_para_dlocal_lista_temp.append(lag_col_name_actual_temp)
predictor_cols_lag1_para_dlocal_lista_temp = sorted(predictor_cols_lag1_para_dlocal_lista_temp)

print(f"Aplicando dropna() usando las {len(predictor_cols_lag1_para_dlocal_lista_temp)} columnas _lag1 generadas...")
subset_dropna_dlocal_actual_temp = [lag_col for lag_col in predictor_cols_lag1_para_dlocal_lista_temp if lag_col in df_dlocal_con_lags_generados_temp.columns]

if not subset_dropna_dlocal_actual_temp:
     print("  ADVERTENCIA: No se generaron columnas de lag válidas para dropna. df_dlocal_features_con_lags será copia sin dropna.")
     df_dlocal_features_con_lags = df_dlocal_con_lags_generados_temp.copy()
else:
    rows_before_dropna_dlocal_temp = len(df_dlocal_con_lags_generados_temp)
    df_dlocal_features_con_lags = df_dlocal_con_lags_generados_temp.dropna(subset=subset_dropna_dlocal_actual_temp).copy()
    rows_after_dropna_dlocal_temp = len(df_dlocal_features_con_lags)
    print(f"  Se eliminaron {rows_before_dropna_dlocal_temp - rows_after_dropna_dlocal_temp} filas por dropna.")

if df_dlocal_features_con_lags.empty:
    raise ValueError("El DataFrame df_dlocal_features_con_lags quedó vacío.")

if 'time_step' not in df_dlocal_features_con_lags.columns:
    df_dlocal_features_con_lags.sort_values(['Empresa', 'Fecha'], inplace=True)
    df_dlocal_features_con_lags['time_step'] = df_dlocal_features_con_lags.groupby('Empresa').cumcount()

print(f"DataFrame 'df_dlocal_features_con_lags' (shape: {df_dlocal_features_con_lags.shape}) está listo para inspección en otra celda.")

cols_id_para_pipeline_dlocal_temp = ['Empresa', 'Fecha', 'time_step']
final_cols_para_input_al_pipeline_X_temp = cols_id_para_pipeline_dlocal_temp + predictor_cols_lag1_para_dlocal_lista_temp
final_cols_para_input_al_pipeline_X_existentes_temp = []
seen_cols_in_X_input_temp = set()
for col_x_input_temp in final_cols_para_input_al_pipeline_X_temp:
    if col_x_input_temp in df_dlocal_features_con_lags.columns and col_x_input_temp not in seen_cols_in_X_input_temp:
        final_cols_para_input_al_pipeline_X_existentes_temp.append(col_x_input_temp)
        seen_cols_in_X_input_temp.add(col_x_input_temp)
    elif col_x_input_temp not in df_dlocal_features_con_lags.columns:
         print(f"  Advertencia: Columna '{col_x_input_temp}' para X del pipeline no está en df_dlocal_features_con_lags.")

print(f"Columnas que conformarán X para el pipeline (antes del DropColumns interno): {len(final_cols_para_input_al_pipeline_X_existentes_temp)} {final_cols_para_input_al_pipeline_X_existentes_temp}")

if len(predictor_cols_lag1_para_dlocal_lista_temp) != n_features_esperadas_por_estimador_modelo:
    raise ValueError(
        f"DISCREPANCIA CRÍTICA DE FEATURES para DLocal: \n"
        f"  - Modelo espera {n_features_esperadas_por_estimador_modelo} features (lags).\n"
        f"  - Se generaron {len(predictor_cols_lag1_para_dlocal_lista_temp)} features _lag1 para DLocal.\n")
print(f"Confirmado: El estimador recibirá {len(predictor_cols_lag1_para_dlocal_lista_temp)} predictores _lag1.")
print(f"  LOS PREDICTORES _lag1 QUE EL ESTIMADOR VERÁ SON: {predictor_cols_lag1_para_dlocal_lista_temp}")
# -------------------------------------------------------------------

# --- 2. PREDICCIÓN ONE-STEP AHEAD RETROSPECTIVA Y FORECAST ---
print("\n" + "="*80)
print("Predicción One-Step Ahead y Forecast para DLocal")
print("="*80)

n_rows_dlocal_pred = len(df_dlocal_features_con_lags)
lista_fechas_pred_dlocal_final = []
lista_predicciones_dlocal_final = []
lista_reales_dlocal_final = []
printed_features_for_predict_flag = False

if n_rows_dlocal_pred < 1:
    print("No hay suficientes filas en df_dlocal_features_con_lags para hacer predicciones.")
else:
    if n_rows_dlocal_pred > 1:
        print(f"Realizando {n_rows_dlocal_pred - 1} predicciones retrospectivas...")
        for i in range(1, n_rows_dlocal_pred):
            fila_input_df_pred = df_dlocal_features_con_lags.iloc[[i-1]]
            X_para_pipeline_pred = fila_input_df_pred[final_cols_para_input_al_pipeline_X_existentes_temp]
            
            if not printed_features_for_predict_flag:
                temp_df_after_drop_pred = loaded_pipeline.named_steps['drop_ids'].transform(X_para_pipeline_pred)
                print(f"\n  --- DEBUG: Features que llegan al estimador (ej. primera predicción) ---")
                print(f"  Columnas en X_para_pipeline_pred (antes del drop interno): {X_para_pipeline_pred.columns.tolist()}")
                print(f"  Columnas DESPUÉS del drop_ids (lo que ve el 'model'): {temp_df_after_drop_pred.columns.tolist()}")
                print(f"  Número de features para el estimador: {temp_df_after_drop_pred.shape[1]}")
                print(f"  -------------------------------------------------------------------\n")
                printed_features_for_predict_flag = True

            fecha_target_pred = df_dlocal_features_con_lags.iloc[i]['Fecha']
            valor_real_target_pred = df_dlocal_features_con_lags.iloc[i]['P_E'] if 'P_E' in df_dlocal_features_con_lags.columns else np.nan

            try:
                pred_raw_val = loaded_pipeline.predict(X_para_pipeline_pred)[0]
                pred_original_val = pred_raw_val
                if use_log_transform_on_target_modelo:
                    with np.errstate(over='ignore', invalid='ignore'): pred_original_val = np.expm1(pred_raw_val)
                    if not np.isfinite(pred_original_val): pred_original_val = np.nan
                lista_predicciones_dlocal_final.append(pred_original_val)
                lista_fechas_pred_dlocal_final.append(fecha_target_pred)
                lista_reales_dlocal_final.append(valor_real_target_pred)
            except Exception as e_pred_loop_final:
                print(f"Error prediciendo para target de fecha {fecha_target_pred}: {e_pred_loop_final}")
                lista_predicciones_dlocal_final.append(np.nan); lista_fechas_pred_dlocal_final.append(fecha_target_pred); lista_reales_dlocal_final.append(valor_real_target_pred)
    else: print("No hay suficientes filas para el backtest retrospectivo.")

    print("Generando forecast para el próximo período...")
    fila_input_forecast_df_pred = df_dlocal_features_con_lags.iloc[[-1]]
    X_para_pipeline_forecast_pred = fila_input_forecast_df_pred[final_cols_para_input_al_pipeline_X_existentes_temp]
    pronostico_siguiente_val = np.nan
    try:
        pred_raw_forecast_val = loaded_pipeline.predict(X_para_pipeline_forecast_pred)[0]
        if use_log_transform_on_target_modelo:
            with np.errstate(over='ignore', invalid='ignore'): pronostico_siguiente_val = np.expm1(pred_raw_forecast_val)
            if not np.isfinite(pronostico_siguiente_val): pronostico_siguiente_val = np.nan
        else: pronostico_siguiente_val = pred_raw_forecast_val
    except Exception as e_forecast_loop_final:
        print(f"Error durante el forecast: {e_forecast_loop_final}")

    fecha_pronostico_siguiente_val = "Periodo Siguiente"
    if not df_dlocal_features_con_lags.empty and 'Fecha' in df_dlocal_features_con_lags.columns:
        try: fecha_pronostico_siguiente_val = df_dlocal_features_con_lags['Fecha'].iloc[-1] + pd.DateOffset(months=1)
        except Exception: pass

    df_backtest_dlocal_resultado = pd.DataFrame({'Fecha': lista_fechas_pred_dlocal_final, 'P_E_Predicho': lista_predicciones_dlocal_final, 'P_E_Real': lista_reales_dlocal_final})
    df_forecast_dlocal_resultado = pd.DataFrame({'Fecha': [fecha_pronostico_siguiente_val], 'P_E_Predicho': [pronostico_siguiente_val]})

    print("\n--- Resultados del Backtest para DLocal ---")
    if df_backtest_dlocal_resultado.empty: print("No se generaron resultados de backtest.")
    else:
        with pd.option_context('display.max_rows', None): print(df_backtest_dlocal_resultado.to_string(float_format="%.4f"))
    print(f"\n--- Pronóstico para {fecha_pronostico_siguiente_val} ---")
    print(df_forecast_dlocal_resultado.to_string(float_format="%.4f"))

    if not df_backtest_dlocal_resultado.empty:
        print("\n--- Métricas del Backtest (DLocal) ---")
        y_real_eval_dlocal_res = df_backtest_dlocal_resultado['P_E_Real'].dropna()
        y_pred_eval_dlocal_res = df_backtest_dlocal_resultado.loc[y_real_eval_dlocal_res.index, 'P_E_Predicho'].dropna()
        y_real_eval_aligned_dlocal_res = y_real_eval_dlocal_res.loc[y_pred_eval_dlocal_res.index]
        if not y_real_eval_aligned_dlocal_res.empty and not y_pred_eval_dlocal_res.empty:
            metricas_dlocal_res = evaluate_model(y_real_eval_aligned_dlocal_res, y_pred_eval_dlocal_res)
            print(f"RMSE: {metricas_dlocal_res['RMSE']:.4f}")
            print(f"MAE:  {metricas_dlocal_res['MAE']:.4f}")
            print(f"R2:   {metricas_dlocal_res['R2']:.4f}")
        else: print("No hay suficientes datos superpuestos para calcular métricas.")
    else: print("No hay datos de backtest para calcular métricas.")

    plt.figure(figsize=(14, 7))
    if not df_backtest_dlocal_resultado.empty:
        plt.plot(df_backtest_dlocal_resultado['Fecha'], df_backtest_dlocal_resultado['P_E_Real'], marker='.', linestyle='-', label='P/E Real DLocal')
        plt.plot(df_backtest_dlocal_resultado['Fecha'], df_backtest_dlocal_resultado['P_E_Predicho'], marker='x', linestyle='--', label='P/E Predicho (Backtest)')
    if isinstance(fecha_pronostico_siguiente_val, pd.Timestamp) and pd.notna(pronostico_siguiente_val):
         plt.scatter([fecha_pronostico_siguiente_val], [pronostico_siguiente_val], color='red', label=f'Forecast {fecha_pronostico_siguiente_val.strftime("%Y-%m")}', zorder=5, s=100)
    plt.title(f'Backtest y Forecast P/E para DLocal (Modelo: {metodo_features_original_modelo}, LogT: {use_log_transform_on_target_modelo})')
    plt.xlabel('Fecha'); plt.ylabel('P_E Ratio'); plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()
# -------------------------------------------------------------------

In [ ]:
# Celda 2: BLOQUE DE INSPECCIÓN DEL DATAFRAME CON LAGS DE DLOCAL

if 'df_dlocal_features_con_lags' in locals() and isinstance(df_dlocal_features_con_lags, pd.DataFrame):
    print("\n" + "="*30 + " Inspección de df_dlocal_features_con_lags " + "="*30)
    if df_dlocal_features_con_lags.empty:
        print("DataFrame 'df_dlocal_features_con_lags' está vacío.")
    else:
        print(f"Shape: {df_dlocal_features_con_lags.shape}")
        print("\nColumnas Presentes:")
        for col_name in df_dlocal_features_con_lags.columns:
            print(f"  - {col_name}")
        
        with pd.option_context('display.max_rows', 10, 
                               'display.max_columns', None, 
                               'display.width', 1500):
            print("\nPrimeras 5 filas de 'df_dlocal_features_con_lags':")
            from IPython.display import display # Para mejor formato en Jupyter
            display(df_dlocal_features_con_lags.head(5))
            
            print("\nÚltimas 5 filas de 'df_dlocal_features_con_lags':")
            display(df_dlocal_features_con_lags.tail(5))
        
        # Descripción de columnas seleccionadas
        # (Asegúrate de que 'columnas_base_efectivas_para_dlocal_lags_temp' y 
        # 'predictor_cols_lag1_para_dlocal_lista_temp' estén disponibles si se definieron
        # en la celda anterior y quieres usarlas aquí. O toma de 'columnas_base_para_lags_modelo')
        
        cols_para_describir_inspeccion = []
        if 'P_E' in df_dlocal_features_con_lags.columns:
            cols_para_describir_inspeccion.append('P_E')
        
        # Añadir algunas columnas base y sus lags correspondientes para comparar
        # Usar 'columnas_base_para_lags_modelo' que fue cargada del paquete
        if 'columnas_base_para_lags_modelo' in locals():
            num_ejemplos_inspeccion = min(3, len(columnas_base_para_lags_modelo))
            for i in range(num_ejemplos_inspeccion):
                base_col_ej_inspeccion = columnas_base_para_lags_modelo[i]
                lag_col_ej_inspeccion = f"{base_col_ej_inspeccion}_lag1"
                if base_col_ej_inspeccion in df_dlocal_features_con_lags.columns:
                    cols_para_describir_inspeccion.append(base_col_ej_inspeccion)
                if lag_col_ej_inspeccion in df_dlocal_features_con_lags.columns:
                    cols_para_describir_inspeccion.append(lag_col_ej_inspeccion)
        
        cols_para_describir_inspeccion = sorted(list(set(cols_para_describir_inspeccion)))

        if cols_para_describir_inspeccion:
            print("\nDescripción de columnas seleccionadas en 'df_dlocal_features_con_lags':")
            with pd.option_context('display.max_columns', None, 'display.width', 1000):
                display(df_dlocal_features_con_lags[cols_para_describir_inspeccion].describe(include='all'))
        
        print("\nConteo de NaNs por columna en 'df_dlocal_features_con_lags' (solo columnas con NaNs):")
        nan_counts_inspeccion = df_dlocal_features_con_lags.isnull().sum()
        print(nan_counts_inspeccion[nan_counts_inspeccion > 0].to_string() if not nan_counts_inspeccion[nan_counts_inspeccion > 0].empty else "No hay NaNs.")

    print("="* (62 + len(" Inspección de df_dlocal_features_con_lags ")))
else:
    print("La variable 'df_dlocal_features_con_lags' no está definida o no es un DataFrame.")
    print("Asegúrate de haber ejecutado la celda anterior (Bloque de Predicción de DLocal) donde se crea.")

In [ ]:
import joblib
import os
import pandas as pd # Para el DataFrame de DLocal
import numpy as np
# Asume que las funciones evaluate_model, etc., y df_dlocal, listas_columnas están definidas

# ... (tu código de carga de model_package_test que ya funciona) ...
# if os.path.exists(ruta_completa_archivo):
#     try:
#         model_package_test = joblib.load(ruta_completa_archivo)
#         # ... (verificaciones) ...
#     except Exception as e:
#         # ... (manejo de error) ...
# else:
#     # ... (manejo de error) ...

# --- ASUMIMOS QUE model_package_test SE CARGÓ Y ES UN DICCIONARIO ---

if isinstance(model_package_test, dict) and 'mejores_pipelines_por_tipo' in model_package_test:
    print("\n--- Ejecutando predicciones para DLocal con el mejor modelo de cada tipo ---")

    metodo_usado_en_paquete = model_package_test.get('metodo_usado', 'desconocido')
    lag_config_paquete = model_package_test.get('lag_config', 'desconocido') # ej. 'lag1', 'lag3', 'lag1_lag3'
    log_transform_paquete = model_package_test.get('log_transform_usado', True) # Default a True si no está

    # Determinar qué lags se usaron para generar las features del modelo
    # Esto es crucial para preparar los datos de DLocal correctamente.
    lags_a_generar_para_dlocal = []
    if lag_config_paquete == 'lag1':
        lags_a_generar_para_dlocal = [1]
    elif lag_config_paquete == 'lag3':
        lags_a_generar_para_dlocal = [3]
    elif lag_config_paquete == 'lag1_lag3':
        lags_a_generar_para_dlocal = [1, 3]
    else:
        # Intenta adivinar por el nombre del archivo o establece un default
        if "LAG3" in nombre_archivo_a_cargar.upper() or "LAG_3" in nombre_archivo_a_cargar.upper():
            lags_a_generar_para_dlocal = [3]
            print(f"Advertencia: lag_config no clara en el paquete, asumiendo lag3 basado en nombre de archivo.")
        elif "LAG1" in nombre_archivo_a_cargar.upper() or "LAG_1" in nombre_archivo_a_cargar.upper():
            lags_a_generar_para_dlocal = [1]
            print(f"Advertencia: lag_config no clara en el paquete, asumiendo lag1 basado en nombre de archivo.")
        else:
            # Si no puedes determinarlo, tendrás que configurarlo manualmente o el preprocesamiento fallará.
            raise ValueError(f"No se pudo determinar la configuración de lag del paquete ('{lag_config_paquete}'). Necesitas saber qué lags usar para DLocal.")

    if not lags_a_generar_para_dlocal:
        raise ValueError("No se especificaron lags para generar para los datos de DLocal.")

    # --- Preparar datos de DLocal UNA VEZ con los lags correctos ---
    # (Esta lógica es similar a tu script de predicción, pero parametrizada por lags_a_generar_para_dlocal)
    print(f"\nPreparando datos de DLocal para método '{metodo_usado_en_paquete}' con lags: {lags_a_generar_para_dlocal}")
    df_dlocal_preparado_full = df_dlocal.copy()
    # 1.1 Conversión Numérica
    columnas_base_paquete = listas_columnas.get(metodo_usado_en_paquete, [])
    if not columnas_base_paquete:
        raise ValueError(f"Método '{metodo_usado_en_paquete}' no encontrado en listas_columnas o no tiene columnas.")
    
    columnas_a_convertir = [c for c in columnas_base_paquete if c not in ['Empresa', 'Fecha', 'P_E']]
    for col in columnas_a_convertir:
        if col in df_dlocal_preparado_full.columns and not pd.api.types.is_numeric_dtype(df_dlocal_preparado_full[col]):
            df_dlocal_preparado_full[col] = pd.to_numeric(df_dlocal_preparado_full[col], errors='coerce')
    
    # 1.2 Imputación Pre-Lag
    cols_imputar_prelag = [c for c in df_dlocal_preparado_full.columns if c not in ['Empresa', 'Fecha']]
    df_dlocal_preparado_full.sort_values(['Empresa', 'Fecha'], inplace=True)
    if df_dlocal_preparado_full['Empresa'].nunique() > 1:
         df_dlocal_preparado_full[cols_imputar_prelag] = df_dlocal_preparado_full.groupby('Empresa')[cols_imputar_prelag].ffill()
    else:
         df_dlocal_preparado_full[cols_imputar_prelag] = df_dlocal_preparado_full[cols_imputar_prelag].ffill()
    df_dlocal_preparado_full.fillna(0, inplace=True)

    # 1.3 Generar Lags
    features_base_sin_pe = [c for c in columnas_base_paquete if c != 'P_E']
    df_dlocal_con_lags_y_ids = df_dlocal_preparado_full[['Empresa', 'Fecha', 'P_E'] + features_base_sin_pe].copy()
    
    generated_lag_feature_names = []
    for lag_val in lags_a_generar_para_dlocal:
        for col_base in features_base_sin_pe:
            if col_base in df_dlocal_con_lags_y_ids.columns:
                lag_col_name = f"{col_base}_lag{lag_val}"
                df_dlocal_con_lags_y_ids[lag_col_name] = df_dlocal_con_lags_y_ids.groupby('Empresa')[col_base].shift(lag_val)
                generated_lag_feature_names.append(lag_col_name)
    
    generated_lag_feature_names = sorted(list(set(generated_lag_feature_names))) # Únicos y ordenados

    # 1.4 Aplicar dropna
    df_dlocal_procesado_con_ids = df_dlocal_con_lags_y_ids.dropna(subset=generated_lag_feature_names).copy()

    if df_dlocal_procesado_con_ids.empty:
        raise ValueError("DataFrame de DLocal vacío después de generar lags y dropna. No se puede predecir.")
    
    print(f"Shape de DLocal procesado (con IDs): {df_dlocal_procesado_con_ids.shape}")
    # --- Fin de preparación de datos de DLocal ---


    for tipo_modelo, pipeline_obj in model_package_test['mejores_pipelines_por_tipo'].items():
        print(f"\n--- Prediciendo con el mejor modelo tipo: {tipo_modelo} ---")
        
        if isinstance(pipeline_obj, str) and pipeline_obj.startswith("Error:"):
            print(f"Error: El pipeline para {tipo_modelo} es un string de error: {pipeline_obj}")
            continue
        if pipeline_obj is None:
            print(f"Error: El pipeline para {tipo_modelo} es None.")
            continue

        # Obtener las features que este pipeline/modelo específico espera
        # (Asumiendo que el pipeline_obj es el modelo final o un Pipeline de scikit-learn
        #  y que fue entrenado SIN DropColumns, por lo que espera solo las features de lag)
        current_model_to_inspect = pipeline_obj
        if hasattr(pipeline_obj, 'steps'): # Si es un Pipeline
            current_model_to_inspect = pipeline_obj.steps[-1][1]

        expected_features_for_this_model = []
        if hasattr(current_model_to_inspect, 'feature_names_in_'):
            expected_features_for_this_model = list(current_model_to_inspect.feature_names_in_)
        elif hasattr(current_model_to_inspect, 'n_features_in_'):
            # Si solo tenemos n_features_in_, asumimos que son las generated_lag_feature_names
            # Esto es una suposición fuerte; idealmente, feature_names_in_ estaría disponible.
            if current_model_to_inspect.n_features_in_ == len(generated_lag_feature_names):
                expected_features_for_this_model = generated_lag_feature_names
            else:
                print(f"ADVERTENCIA para {tipo_modelo}: n_features_in_ ({current_model_to_inspect.n_features_in_}) no coincide con el número de lags generados ({len(generated_lag_feature_names)}). Saltando.")
                continue
        else:
            print(f"ADVERTENCIA para {tipo_modelo}: No se pueden determinar las features esperadas. Usando todos los lags generados.")
            expected_features_for_this_model = generated_lag_feature_names
        
        if not expected_features_for_this_model:
            print(f"Error: No se pudieron determinar las features para el modelo {tipo_modelo}.")
            continue

        # Asegurarse de que todas las features esperadas estén en df_dlocal_procesado_con_ids
        missing_cols = [col for col in expected_features_for_this_model if col not in df_dlocal_procesado_con_ids.columns]
        if missing_cols:
            print(f"Error para {tipo_modelo}: Faltan columnas en los datos de DLocal: {missing_cols}. Saltando.")
            continue
            
        # Preparar X_pred para este modelo (solo las features que espera)
        df_dlocal_model_input = df_dlocal_procesado_con_ids[expected_features_for_this_model].copy()
        df_dlocal_eval_data = df_dlocal_procesado_con_ids[['Fecha', 'P_E']].copy() # Para P/E real y fechas

        if df_dlocal_model_input.empty:
            print(f"DataFrame de input para el modelo {tipo_modelo} está vacío. Saltando.")
            continue

        # --- Lógica de Predicción (similar a tu script original) ---
        n_rows_for_pred = len(df_dlocal_model_input)
        fechas_pred_actual = []
        predicciones_actual = []
        valores_reales_actual = []

        if n_rows_for_pred > 0: # Necesitas al menos una fila para el forecast, y >1 para backtest
            # Bucle retrospectivo (si hay al menos 2 filas para predecir la segunda en adelante)
            if n_rows_for_pred > 1:
                print(f"  Realizando {n_rows_for_pred-1} predicciones retrospectivas para {tipo_modelo}...")
                for i in range(1, n_rows_for_pred):
                    X_pred_row = df_dlocal_model_input.iloc[[i-1]]
                    try:
                        pred_raw = pipeline_obj.predict(X_pred_row)[0]
                        if log_transform_paquete:
                            with np.errstate(over='ignore', invalid='ignore'): pred_orig = np.expm1(pred_raw)
                            if not np.isfinite(pred_orig): pred_orig = np.nan
                        else:
                            pred_orig = pred_raw
                        predicciones_actual.append(pred_orig)
                        fechas_pred_actual.append(df_dlocal_eval_data.iloc[i]['Fecha'])
                        valores_reales_actual.append(df_dlocal_eval_data.iloc[i]['P_E'])
                    except Exception as e_pred:
                        print(f"    Error prediciendo para {tipo_modelo} (Fecha: {df_dlocal_eval_data.iloc[i]['Fecha']}): {e_pred}")
                        predicciones_actual.append(np.nan)
                        fechas_pred_actual.append(df_dlocal_eval_data.iloc[i]['Fecha'])
                        valores_reales_actual.append(df_dlocal_eval_data.iloc[i]['P_E'])
            
            # Forecast para el siguiente período
            print(f"  Generando forecast para {tipo_modelo}...")
            X_pred_ultimo = df_dlocal_model_input.iloc[[-1]]
            pronostico_next_actual = np.nan
            try:
                pronostico_next_raw = pipeline_obj.predict(X_pred_ultimo)[0]
                if log_transform_paquete:
                    with np.errstate(over='ignore', invalid='ignore'): pronostico_next_actual = np.expm1(pronostico_next_raw)
                    if not np.isfinite(pronostico_next_actual): pronostico_next_actual = np.nan
                else:
                    pronostico_next_actual = pronostico_next_raw
            except Exception as e_forecast:
                print(f"    Error durante el forecast para {tipo_modelo}: {e_forecast}")

            try:
                proxima_fecha_actual = df_dlocal_eval_data['Fecha'].iloc[-1] + pd.DateOffset(months=1)
            except:
                proxima_fecha_actual = "Periodo Siguiente"

            df_backtest_actual = pd.DataFrame({'Fecha': fechas_pred_actual, 'P_E_Predicho': predicciones_actual, 'P_E_Real': valores_reales_actual})
            df_forecast_actual = pd.DataFrame({'Fecha': [proxima_fecha_actual], f'P_E_Predicho_{tipo_modelo}': [pronostico_next_actual]})

            print(f"\n  --- Resultados del Backtest para DLocal (Modelo: {tipo_modelo}) ---")
            if not df_backtest_actual.empty:
                with pd.option_context('display.max_rows', 10): print(df_backtest_actual.to_string(float_format="%.4f"))
                metricas_dlocal_actual = evaluate_model(df_backtest_actual['P_E_Real'], df_backtest_actual['P_E_Predicho'])
                print(f"  Métricas (Modelo: {tipo_modelo}): RMSE={metricas_dlocal_actual['RMSE']:.4f}, MAE={metricas_dlocal_actual['MAE']:.4f}, R2={metricas_dlocal_actual['R2']:.4f}")
            else:
                print("  No hay datos de backtest para este modelo.")
            
            print(f"\n  --- Pronóstico para {proxima_fecha_actual} (Modelo: {tipo_modelo}) ---")
            print(df_forecast_actual.to_string(float_format="%.4f"))

            # Graficar (opcional, o podrías acumular y graficar todos juntos)
            if not df_backtest_actual.empty:
                plt.figure(figsize=(12, 6))
                plt.plot(df_backtest_actual['Fecha'], df_backtest_actual['P_E_Real'], marker='.', linestyle='-', label='P/E Real DLocal')
                plt.plot(df_backtest_actual['Fecha'], df_backtest_actual['P_E_Predicho'], marker='x', linestyle='--', label=f'P/E Predicho ({tipo_modelo})')
                if isinstance(proxima_fecha_actual, pd.Timestamp) and pd.notna(pronostico_next_actual):
                     plt.scatter([proxima_fecha_actual], [pronostico_next_actual], color='red', label=f'Forecast {tipo_modelo} {proxima_fecha_actual.strftime("%Y-%m")}', zorder=5, s=70)
                plt.title(f'Backtest y Forecast P/E DLocal (Modelo: {metodo_usado_en_paquete} - {tipo_modelo}, Lags: {lags_a_generar_para_dlocal})')
                plt.xlabel('Fecha'); plt.ylabel('P/E Ratio'); plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()
        else:
            print(f"  No hay suficientes datos en DLocal procesado para hacer predicciones con {tipo_modelo}.")

elif isinstance(model_package_test, dict):
    print("ADVERTENCIA: La clave 'mejores_pipelines_por_tipo' NO está presente en el diccionario cargado.")
    print(f"Claves disponibles: {list(model_package_test.keys())}")
    print("No se pueden ejecutar los modelos por tipo.")
else:
    print(f"ADVERTENCIA: El archivo cargado NO es un diccionario o no se pudo cargar. Es de tipo: {type(model_package_test)}")

#### Sin transformación logarítmica

In [ ]:
# Celda: BLOQUE DE EXPERIMENTOS (ENTRENAR SOLO CON LAGS Y GUARDAR PAQUETE POR MÉTODO)

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import joblib
import os # Para os.path.abspath

# --- Modelos ---
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# --- Métricas y Preprocesamiento ---
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer
# from sklearn.base import BaseEstimator, TransformerMixin # No se usa si DropColumns no está en pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, GroupKFold # BaseCrossValidator no se usa directamente
from sklearn.pipeline import Pipeline

# Ignorar warnings comunes (opcional)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
pd.options.mode.chained_assignment = None

# =====================================
# FUNCIONES AUXILIARES (Asegúrate de que estén definidas como en tu script original)
# =====================================
def rmse(y_true, y_pred):
    y_true_np, y_pred_np = np.asarray(y_true), np.asarray(y_pred)
    mask = np.isfinite(y_true_np) & np.isfinite(y_pred_np)
    if not np.all(mask): y_true_np, y_pred_np = y_true_np[mask], y_pred_np[mask]
    if len(y_true_np) == 0: return np.nan
    return np.sqrt(mean_squared_error(y_true_np, y_pred_np))

rmse_scorer = make_scorer(rmse, greater_is_better=False)

def train_test_split_by_empresa(df_model_input, n_empresas_prueba):
    df_model = df_model_input.copy()
    if 'Empresa' not in df_model.columns: raise KeyError("'Empresa' necesaria")
    if 'P_E' not in df_model.columns: raise KeyError("'P_E' (target) necesaria")
    empresas_unicas = df_model['Empresa'].unique()
    n_total_empresas = len(empresas_unicas)
    if n_total_empresas == 0: return None, None, None, None
    if n_total_empresas == 1:
        X = df_model.drop(columns=['P_E']); y = df_model['P_E']
        return X.copy(), y.copy(), pd.DataFrame(columns=X.columns), pd.Series(dtype=y.dtype)
    n_empresas_prueba_valido = max(0, min(n_empresas_prueba, n_total_empresas - 1 if n_total_empresas > 0 else 0))
    if n_empresas_prueba_valido != n_empresas_prueba:
        print(f"  Advertencia train_test_split: n_empresas_prueba ajustado de {n_empresas_prueba} a {n_empresas_prueba_valido}")
    n_empresas_prueba = n_empresas_prueba_valido
    if n_empresas_prueba == 0:
        empresas_entrenamiento = empresas_unicas; empresas_prueba = np.array([])
    else:
        empresas_entrenamiento, empresas_prueba = train_test_split(
            empresas_unicas, test_size=n_empresas_prueba, random_state=42, shuffle=True)
    mask_entrenamiento = df_model['Empresa'].isin(empresas_entrenamiento)
    X = df_model.drop(columns=['P_E']); y = df_model['P_E']
    X_train, y_train = X[mask_entrenamiento].copy(), y[mask_entrenamiento].copy()
    if n_empresas_prueba > 0 and len(empresas_prueba) > 0:
        mask_prueba = df_model['Empresa'].isin(empresas_prueba)
        X_test, y_test = X[mask_prueba].copy(), y[mask_prueba].copy()
        if X_test.empty and not mask_prueba.empty() and not X_train.empty : 
             X_test = pd.DataFrame(columns=X_train.columns)
             y_test = pd.Series(dtype=y_train.dtype)
    else:
        X_test_cols = X_train.columns if not X_train.empty else (X.columns if not X.empty else [])
        y_test_dtype = y_train.dtype if not y_train.empty else (y.dtype if not y.empty else float)
        X_test, y_test = pd.DataFrame(columns=X_test_cols), pd.Series(dtype=y_test_dtype)
    if X_train.empty and n_total_empresas > 0 :
        return None, None, None, None
    return X_train, y_train, X_test, y_test

def evaluate_model(y_test, y_pred):
    y_test_np, y_pred_np = np.asarray(y_test), np.asarray(y_pred)
    mask = np.isfinite(y_test_np) & np.isfinite(y_pred_np)
    if not np.all(mask): y_test_np, y_pred_np = y_test_np[mask], y_pred_np[mask]
    if len(y_test_np) == 0: return {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}
    mae = mean_absolute_error(y_test_np, y_pred_np)
    rmse_val = rmse(y_test_np, y_pred_np)
    r2 = np.nan
    if len(y_test_np) > 1 and np.var(y_test_np) > 1e-9:
        try: r2 = r2_score(y_test_np, y_pred_np)
        except ValueError: pass
    elif len(y_test_np) > 0 and mean_squared_error(y_test_np, y_pred_np) < 1e-9: r2 = 1.0
    elif len(y_test_np) > 0: r2 = 0.0
    return {'RMSE': rmse_val, 'MAE': mae, 'R2': r2}

def sort_by_timestep(X_df, y_series):
    X, y = X_df.copy(), y_series.copy()
    if 'time_step' not in X.columns: return X, y
    if not isinstance(y, pd.Series) or not X.index.equals(y.index):
        if len(y) == len(X): y = pd.Series(np.asarray(y), index=X.index, name=getattr(y, 'name', 'target'))
        else: print("Error sort_by_timestep: y no alineable con X."); return X, y
    X_sorted = X.sort_values('time_step'); y_sorted = y.loc[X_sorted.index]
    return X_sorted, y_sorted

# ================================================================================
# BLOQUE DE EXPERIMENTOS (ENTRENAR SOLO CON CARACTERÍSTICAS REZAGADAS '_lag1')
# ================================================================================
# ASUNCIONES:
# 1. `dfs_shifted`: Diccionario {metodo: DataFrame_con_features_lag1_y_IDs_y_target}
#    Este DataFrame es el resultado del "Bloque 8" que genera los lags.
#    Debe contener 'Empresa', 'Fecha', 'P_E', y las columnas `_lag1` que se usarán como predictores.
#    También puede contener 'time_step' y las columnas base originales (no se usarán como predictores).

results_accumulator = [] # Lista para acumular TODOS los resultados de TODOS los métodos
start_time_total_experiments = time.time()
use_log_transform_experiment = False # Configuración global para este bloque de experimentos
test_size_ratio_experiment = 0.20
LAG_SUFFIX_TO_USE = '_lag1' # Define qué sufijo de lag buscar para los predictores

# --- Iterar sobre los métodos de features ---
for metodo_features, df_original_con_lags_ids_target in dfs_shifted.items():
    start_time_metodo = time.time()
    print("\n" + "="*80)
    print(f"Procesando Método de Features: {metodo_features} {'(CON Log Transform)' if use_log_transform_experiment else '(SIN Log Transform)'} (Predictors: {LAG_SUFFIX_TO_USE})")
    print("="*80 + "\n")

    if df_original_con_lags_ids_target.empty:
        print(f"  ⇨ ADVERTENCIA: DataFrame de entrada para el método {metodo_features} está vacío. Omitiendo.")
        # Podrías añadir una entrada de error a results_accumulator si lo deseas
        continue

    df_current_method_processed = df_original_con_lags_ids_target.copy()

    if 'time_step' not in df_current_method_processed.columns:
        df_current_method_processed.sort_values(['Empresa', 'Fecha'], inplace=True)
        df_current_method_processed['time_step'] = df_current_method_processed.groupby('Empresa').cumcount()

    # Identificar las columnas de predictores (solo las que terminan con el sufijo de lag especificado)
    predictor_cols_for_model = sorted([c for c in df_current_method_processed.columns if c.endswith(LAG_SUFFIX_TO_USE)])
    
    if not predictor_cols_for_model:
        print(f"  ⇨ ADVERTENCIA: No se encontraron columnas '{LAG_SUFFIX_TO_USE}' en dfs_shifted['{metodo_features}']. Omitiendo método.")
        continue
    
    columnas_base_que_generaron_lags = sorted(list(set(lag_col.replace(LAG_SUFFIX_TO_USE, '') for lag_col in predictor_cols_for_model)))
    print(f"  ⇨ Método {metodo_features}: Se usarán {len(predictor_cols_for_model)} predictores ({LAG_SUFFIX_TO_USE}).")
    print(f"    Lags generados de {len(columnas_base_que_generaron_lags)} columnas base: {columnas_base_que_generaron_lags[:5]}...")

    # df_model contendrá IDs, target y SOLO los predictores de lag seleccionados
    cols_for_df_model_construction = ['Empresa', 'Fecha', 'time_step', 'P_E'] + predictor_cols_for_model
    # Asegurar que solo se usen columnas existentes y únicas
    final_cols_for_df_model_construction = sorted(list(set(c for c in cols_for_df_model_construction if c in df_current_method_processed.columns)))
    
    essential_cols_check = ['Empresa', 'Fecha', 'time_step', 'P_E']
    if not all(ec in final_cols_for_df_model_construction for ec in essential_cols_check):
        missing_ess_cols = [ec for ec in essential_cols_check if ec not in final_cols_for_df_model_construction]
        print(f"  ⇨ ERROR: Faltan columnas esenciales ({missing_ess_cols}) para construir df_model para {metodo_features}. Omitiendo.")
        continue
        
    df_model_for_splitting = df_current_method_processed[final_cols_for_df_model_construction].copy()

    # --- Split Train/Test (X_train_with_ids, X_test_with_ids AÚN TIENEN IDs) ---
    n_total_empresas = df_model_for_splitting['Empresa'].nunique()
    n_empresas_test = 0
    if n_total_empresas > 1:
        n_empresas_test = max(1, int(round(n_total_empresas * test_size_ratio_experiment)))
        if n_total_empresas - n_empresas_test < 1 : n_empresas_test = n_total_empresas - 1 # Dejar al menos 1 para entrenar
    
    X_train_with_ids, y_train_original, X_test_with_ids, y_test_original = train_test_split_by_empresa(df_model_for_splitting, n_empresas_test)
    
    if X_train_with_ids is None or X_train_with_ids.empty:
        print(f"  Split train/test fallido o X_train vacío para {metodo_features}. Omitiendo método.")
        continue
    
    # --- Preparar X para el MODELO (SOLO las columnas predictor_cols_for_model) ---
    X_train_model_features = X_train_with_ids[predictor_cols_for_model].copy()
    X_test_model_features = pd.DataFrame(columns=predictor_cols_for_model) # Inicializar vacío
    if X_test_with_ids is not None and not X_test_with_ids.empty:
        X_test_model_features = X_test_with_ids[predictor_cols_for_model].copy()
    
    # --- Transformar Target y Manejar NaNs/Infs ---
    y_train_transformed_target = y_train_original.copy()
    if use_log_transform_experiment:
        y_numeric = pd.to_numeric(y_train_original, errors='coerce')
        # Usar mediana para imputar NaNs en y_train ANTES de log, y asegurar no negativos
        median_y_train = y_numeric.median() if not y_numeric.isnull().all() else 0.0
        y_numeric = y_numeric.fillna(median_y_train)
        y_numeric.loc[y_numeric < 0] = 0 # Evitar errores con log1p
        y_train_transformed_target = np.log1p(y_numeric)
        # Rellenar NaNs que puedan surgir de log1p(valor muy negativo -> -1 -> 0 -> log1p(0)=0)
        # o si y_numeric original era NaN y median_y_train era tal que log1p dio NaN (improbable con fillna(0) para negativos)
        if y_train_transformed_target.isnull().any():
             median_y_transformed = y_train_transformed_target.median() if not y_train_transformed_target.isnull().all() else 0.0
             y_train_transformed_target = y_train_transformed_target.fillna(median_y_transformed)

    # --- Ordenar Datos (X_train_with_ids para CV, X_train_model_features para fit) ---
    # Asegurar alineación de índices después de cualquier manipulación de y_train_transformed_target
    if not X_train_with_ids.index.equals(y_train_transformed_target.index):
        common_idx_train = X_train_with_ids.index.intersection(y_train_transformed_target.index)
        X_train_with_ids = X_train_with_ids.loc[common_idx_train]
        X_train_model_features = X_train_model_features.loc[common_idx_train]
        y_train_transformed_target = y_train_transformed_target.loc[common_idx_train]
        if X_train_model_features.empty : print(f"X_train_model_features vacío tras realinear para {metodo_features}. Omitiendo."); continue

    X_train_cv_sorted, y_train_cv_sorted = sort_by_timestep(X_train_with_ids, y_train_transformed_target)
    X_train_fit_sorted, y_train_fit_sorted = sort_by_timestep(X_train_model_features, y_train_transformed_target) # y_train_fit_sorted es igual a y_train_cv_sorted

    X_test_eval_sorted, y_test_eval_original_sorted = pd.DataFrame(columns=X_test_with_ids.columns), pd.Series(dtype=y_test_original.dtype)
    X_test_predict_sorted = pd.DataFrame(columns=X_test_model_features.columns)
    if X_test_with_ids is not None and not X_test_with_ids.empty:
        X_test_eval_sorted, y_test_eval_original_sorted = sort_by_timestep(X_test_with_ids, y_test_original)
        X_test_predict_sorted, _ = sort_by_timestep(X_test_model_features, y_test_original)


    # --- Grids y Scenarios (igual que tu última versión) ---
    param_grid_rf   = {'model__max_depth':[10, 20, None],'model__min_samples_split':[5, 10],'model__n_estimators':[100, 200]}
    param_grid_xgb  = {'model__n_estimators':[100, 200],'model__max_depth':[3, 5, 7],'model__learning_rate':[0.1, 0.05]}
    param_grid_lgbm = {'model__n_estimators':[100, 200],'model__learning_rate':[0.1, 0.05],'model__max_depth':[3, 5, 7],'model__num_leaves':[15, 31]}
    param_grid_cb   = {'model__iterations':[100, 200],'model__learning_rate':[0.1, 0.05],'model__depth':[3, 5, 7],'model__l2_leaf_reg':[1, 3]}

    scenarios = { # Usar 'model' como prefijo para HP
        'RF_Simple':  {'model': RandomForestRegressor(random_state=42, n_jobs=-1), 'hp': False, 'vt': False, 'prefix': 'model'},
        'RF_HP':      {'model': RandomForestRegressor(random_state=42, n_jobs=-1), 'hp': True,  'vt': False, 'grid': param_grid_rf, 'prefix': 'model'},
        'XGB_Simple': {'model': XGBRegressor(random_state=42, n_jobs=-1), 'hp': False, 'vt': False, 'prefix': 'model'},
        'XGB_HP':     {'model': XGBRegressor(random_state=42, n_jobs=-1), 'hp': True,  'vt': False, 'grid': param_grid_xgb, 'prefix': 'model'},
        'LGBM_Simple':{'model': LGBMRegressor(random_state=42, n_jobs=-1, verbosity=-1), 'hp': False, 'vt': False, 'prefix': 'model'},
        'LGBM_HP':    {'model': LGBMRegressor(random_state=42, n_jobs=-1, verbosity=-1), 'hp': True,  'vt': False, 'grid': param_grid_lgbm, 'prefix': 'model'},
        'CB_Simple':  {'model': CatBoostRegressor(random_state=42, verbose=0, allow_writing_files=False), 'hp': False, 'vt': False, 'prefix': 'model'},
        'CB_HP':      {'model': CatBoostRegressor(random_state=42, verbose=0, allow_writing_files=False), 'hp': True,  'vt': False, 'grid': param_grid_cb, 'prefix': 'model'},
    }
    # Eliminados escenarios _TV y _TVHP para simplificar, ya que vt=True no se estaba usando consistentemente.
    # Si quieres validación temporal (GroupKFold), vt debería ser True.

    # --- Bucle de Escenarios ---
    for escenario_nombre, escenario_cfg in scenarios.items():
        tiempo_esc_inicio = time.time()
        print(f"\n  --- Escenario: {escenario_nombre} para Método: {metodo_features} ---")
        
        current_base_model = escenario_cfg['model']
        perform_hp_search = escenario_cfg['hp']
        use_temporal_cv_in_scenario = escenario_cfg['vt'] # vt=True usaría GroupKFold

        # Datos para CV (pueden tener IDs si use_temporal_cv_in_scenario y GroupKFold)
        X_cv_input_data = X_train_cv_sorted if use_temporal_cv_in_scenario else X_train_with_ids
        y_cv_input_data = y_train_cv_sorted if use_temporal_cv_in_scenario else y_train_transformed_target
        
        # Datos para FIT (siempre sin IDs, solo features de lag)
        X_fit_data = X_train_fit_sorted if use_temporal_cv_in_scenario else X_train_model_features
        y_fit_data = y_train_fit_sorted if use_temporal_cv_in_scenario else y_train_transformed_target

        # Datos para EVALUACIÓN
        X_eval_data_predict = X_test_predict_sorted if use_temporal_cv_in_scenario else X_test_model_features
        y_eval_data_original_final = y_test_eval_original_sorted if use_temporal_cv_in_scenario else y_test_original

        if X_fit_data.empty or y_fit_data.empty:
            print(f"    ERROR: Datos de entrenamiento (X_fit_data o y_fit_data) vacíos para {escenario_nombre}. Omitiendo.")
            # ... (añadir resultado de error a results_accumulator) ...
            results_accumulator.append({
                'Modelo': escenario_nombre.split('_')[0], 'Validacion_Temp': use_temporal_cv_in_scenario,
                'Busqueda_HP': perform_hp_search, 'Config_Key': metodo_features, 'Metodo': metodo_features,
                'Lag_Config': LAG_SUFFIX_TO_USE, 'Log_Transform': use_log_transform_experiment,
                'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan, 'Best_Params': None,
                'pipeline': 'Error: Datos de entrenamiento vacíos', 
                'columnas_base_usadas': columnas_base_que_generaron_lags,
                'n_features_modelo': len(predictor_cols_for_model), 'Error': 'Datos de entrenamiento vacíos'})
            continue

        # Pipeline para GridSearchCV (solo el modelo con prefijo) o modelo base para fit directo
        # NO HAY DropColumns aquí
        pipeline_for_training = Pipeline([(escenario_cfg['prefix'], current_base_model)]) if perform_hp_search else current_base_model
        
        final_fitted_model_object, hp_best_params_found, error_msg_escenario = None, None, None

        try:
            if perform_hp_search:
                cv_strategy_obj, cv_fit_params_dict = None, {}
                can_perform_cv = True
                
                # X_cv_input_data (con IDs) se usa para generar grupos para GroupKFold
                # X_fit_data (sin IDs) se usa para el fit real dentro de GridSearchCV
                n_groups_for_cv = X_cv_input_data['Empresa'].nunique() if use_temporal_cv_in_scenario and 'Empresa' in X_cv_input_data.columns else 0

                if use_temporal_cv_in_scenario and n_groups_for_cv >= 2:
                    groups_for_gkf = X_cv_input_data['Empresa']
                    n_splits_gkf = min(3, n_groups_for_cv); n_splits_gkf = max(2, n_splits_gkf)
                    if len(X_fit_data) < n_splits_gkf : can_perform_cv = False
                    else: cv_strategy_obj = GroupKFold(n_splits=n_splits_gkf); cv_fit_params_dict = {'groups': groups_for_gkf}
                elif len(X_fit_data) >= 2: # KFold estándar
                    n_samples_kf = len(X_fit_data)
                    kfold_splits_val = min(3, n_samples_kf); kfold_splits_val = max(2, kfold_splits_val)
                    if n_samples_kf < kfold_splits_val : can_perform_cv = False
                    else: cv_strategy_obj = KFold(n_splits=kfold_splits_val, shuffle=True, random_state=42)
                else: can_perform_cv = False
                
                if not can_perform_cv or cv_strategy_obj is None:
                    print(f"    Advertencia: No se puede realizar CV para {escenario_nombre}. Se entrenará modelo simple.")
                    perform_hp_search = False # Forzar fit simple
                
                if perform_hp_search: # Re-chequear por si se cambió a False
                    gs = GridSearchCV(pipeline_for_training, escenario_cfg['grid'], scoring=rmse_scorer, 
                                      cv=cv_strategy_obj, n_jobs=-1, refit=True, error_score='raise', verbose=0)
                    print(f"    Iniciando GridSearchCV con {type(cv_strategy_obj).__name__}...")
                    gs.fit(X_fit_data, y_fit_data, **cv_fit_params_dict) # X_fit_data no tiene IDs
                    final_fitted_model_object, hp_best_params_found = gs.best_estimator_, gs.best_params_
                    print(f"    Mejores parámetros: {hp_best_params_found}")

            if not perform_hp_search or final_fitted_model_object is None: # Si HP se omitió o falló
                print(f"    Entrenando modelo simple para {escenario_nombre}...")
                # Si pipeline_for_training era para HP, ahora usamos current_base_model directamente
                model_to_fit_simple = current_base_model 
                model_to_fit_simple.fit(X_fit_data, y_fit_data)
                final_fitted_model_object = model_to_fit_simple
                hp_best_params_found = None # No hubo HP search
        
        except Exception as e:
            error_msg_escenario = str(e)
            print(f"    ERROR durante entrenamiento/HP search para {escenario_nombre}: {e}")

        # --- Evaluación ---
        eval_metrics_dict, n_features_in_final_model = {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}, "N/A"
        if final_fitted_model_object:
            try:
                # Obtener el estimador real del pipeline (si es un pipeline de GridSearchCV) o el modelo mismo
                actual_estimator = final_fitted_model_object
                if hasattr(final_fitted_model_object, 'steps'): # Es un Pipeline
                    actual_estimator = final_fitted_model_object.steps[-1][1]
                
                n_features_in_final_model = getattr(actual_estimator, 'n_features_in_', "N/A")
                # print(f"    >> Estimador final '{type(actual_estimator).__name__}' vio {n_features_in_final_model} features.")
                # if isinstance(n_features_in_final_model, int) and n_features_in_final_model != len(predictor_cols_for_model):
                #      print(f"       ¡ADVERTENCIA! n_features ({n_features_in_final_model}) no coincide con num. de lags ({len(predictor_cols_for_model)}).")

                if X_eval_data_predict is not None and not X_eval_data_predict.empty and \
                   y_eval_data_original_final is not None and not y_eval_data_original_final.empty:
                    
                    pred_transformed_eval = final_fitted_model_object.predict(X_eval_data_predict)
                    pred_original_scale_eval = np.expm1(pred_transformed_eval) if use_log_transform_experiment else pred_transformed_eval
                    if use_log_transform_experiment: pred_original_scale_eval[~np.isfinite(pred_original_scale_eval)] = np.nan # Manejar Infs de expm1
                    
                    eval_metrics_dict = evaluate_model(y_eval_data_original_final, pred_original_scale_eval)
                    print(f"    Métricas Test {escenario_nombre}: RMSE={eval_metrics_dict['RMSE']:.4f}, MAE={eval_metrics_dict['MAE']:.4f}, R2={eval_metrics_dict['R2']:.4f}")
                else:
                    print(f"    No hay datos de X_eval ({X_eval_data_predict.shape if X_eval_data_predict is not None else 'None'}) / y_eval ({y_eval_data_original_final.shape if y_eval_data_original_final is not None else 'None'}) para evaluar.")
            except Exception as e_eval:
                error_msg_escenario = (error_msg_escenario + f" | EvalError: {str(e_eval)}") if error_msg_escenario else f"EvalError: {str(e_eval)}"
                print(f"    ERROR durante evaluación para {escenario_nombre}: {e_eval}")
        
        results_accumulator.append({
            'Modelo': escenario_nombre.split('_')[0], 
            'Validacion_Temp': use_temporal_cv_in_scenario, # Si se usó GroupKFold
            'Busqueda_HP': escenario_cfg['hp'] and perform_hp_search, # Si HP se intentó y se completó
            'Config_Key': metodo_features, 
            'Metodo': metodo_features,
            'Lag_Config': LAG_SUFFIX_TO_USE + '_solo_pred', 
            'Log_Transform': use_log_transform_experiment, 
            'RMSE': eval_metrics_dict['RMSE'],
            'MAE': eval_metrics_dict['MAE'], 
            'R2': eval_metrics_dict['R2'], 
            'Best_Params': hp_best_params_found,
            'pipeline': final_fitted_model_object, # El objeto modelo/pipeline entrenado
            'columnas_base_usadas': columnas_base_que_generaron_lags,
            'n_features_modelo': n_features_in_final_model, 
            'Error': error_msg_escenario
        })
        print(f"    Tiempo escenario {escenario_nombre}: {time.time() - tiempo_esc_inicio:.2f}s")
    # --- Fin del bucle de escenarios ---
    print(f"\nTiempo total para Método de Features {metodo_features}: {time.time() - start_time_metodo:.2f}s")
# --- Fin del bucle de métodos de features ---
print(f"\n\nTiempo total de ejecución de todos los experimentos: {(time.time() - start_time_total_experiments)/60:.2f} minutos")


# ================================================================================
# CREAR TABLA DE RESULTADOS GENERAL Y GUARDAR PAQUETES POR MÉTODO
# ================================================================================
if results_accumulator:
    df_results_all_experiments = pd.DataFrame(results_accumulator)
    print("\n" + "="*80 + "\n--- Resultados Finales Globales (Modelos Entrenados SOLO CON LAGS) ---\n" + f"Total resultados generados: {len(df_results_all_experiments)}\n")
    
    cols_to_display_summary = ['Metodo', 'Modelo', 'Busqueda_HP', 'RMSE', 'MAE', 'R2', 'n_features_modelo', 'Error', 'Best_Params']
    final_cols_to_display_summary = [col for col in cols_to_display_summary if col in df_results_all_experiments.columns]
    with pd.option_context('display.max_rows', 200, 'display.max_columns', None, 'display.width', 2000, 'display.float_format', '{:.4f}'.format):
        print(df_results_all_experiments.sort_values(by=['Metodo','RMSE'], ascending=[True,True])[final_cols_to_display_summary].to_string(index=False))

    # --- GUARDAR PAQUETE CON MEJORES MODELOS POR TIPO PARA CADA MÉTODO DE FEATURES ---
    for metodo_actual_features in df_results_all_experiments['Metodo'].unique():
        print("\n" + "="*70)
        print(f"--- Preparando paquete de mejores modelos por tipo para Método de Features: {metodo_actual_features} ---")
        
        df_resultados_metodo_actual = df_results_all_experiments[
            (df_results_all_experiments['Metodo'] == metodo_actual_features) &
            (df_results_all_experiments['RMSE'].notna()) &
            (df_results_all_experiments['pipeline'].notna()) & 
            (df_results_all_experiments['Error'].isna()) # Solo resultados sin errores de entrenamiento/eval
        ].copy()

        if df_resultados_metodo_actual.empty:
            print(f"No hay resultados válidos para el método '{metodo_actual_features}' para crear un paquete.")
            continue

        mejores_modelos_info_por_tipo = {}
        modelos_base_a_considerar = ['RF', 'XGB', 'LGBM', 'CB']
        
        # Tomar metadata del primer resultado válido para este método
        # (asumiendo que Log_Transform, columnas_base_usadas, Lag_Config son consistentes para el método)
        metadata_ref_row = df_resultados_metodo_actual.iloc[0]
        log_transform_paquete_actual = metadata_ref_row.get('Log_Transform', True)
        columnas_base_paquete_actual = metadata_ref_row.get('columnas_base_usadas', [])
        lag_config_paquete_actual = metadata_ref_row.get('Lag_Config', LAG_SUFFIX_TO_USE + '_solo_pred')


        for tipo_modelo in modelos_base_a_considerar:
            df_tipo_especifico = df_resultados_metodo_actual[df_resultados_metodo_actual['Modelo'] == tipo_modelo]
            
            if not df_tipo_especifico.empty:
                mejor_fila_tipo_actual = df_tipo_especifico.sort_values(by='RMSE', ascending=True).iloc[0]
                
                mejores_modelos_info_por_tipo[tipo_modelo] = {
                    'pipeline_object': mejor_fila_tipo_actual['pipeline'], # El modelo/pipeline entrenado
                    'RMSE': mejor_fila_tipo_actual['RMSE'],
                    'MAE': mejor_fila_tipo_actual['MAE'],
                    'R2': mejor_fila_tipo_actual['R2'],
                    'Best_Params': mejor_fila_tipo_actual['Best_Params'],
                    'n_features_modelo': mejor_fila_tipo_actual['n_features_modelo'],
                    'Escenario_Validacion_Temp': mejor_fila_tipo_actual['Validacion_Temp'],
                    'Escenario_Busqueda_HP': mejor_fila_tipo_actual['Busqueda_HP']
                }
                print(f"  Mejor {tipo_modelo} para {metodo_actual_features}: RMSE = {mejor_fila_tipo_actual['RMSE']:.4f}, n_features={mejor_fila_tipo_actual['n_features_modelo']}")
            else:
                print(f"  No se encontraron modelos válidos para {tipo_modelo} para el método {metodo_actual_features}.")
                mejores_modelos_info_por_tipo[tipo_modelo] = None 

        if any(mejores_modelos_info_por_tipo.values()): # Si se encontró al menos un mejor modelo
            paquete_final_a_guardar = {
                'metodo_features': metodo_actual_features,
                'lag_config_usada': lag_config_paquete_actual,
                'log_transform_target': log_transform_paquete_actual,
                'columnas_base_para_lags': columnas_base_paquete_actual,
                'mejores_modelos_info_por_tipo': mejores_modelos_info_por_tipo
            }
            
            nombre_archivo_paquete_final = f"paquete_mejores_por_tipo_{metodo_actual_features.replace(' ', '_')}_{lag_config_paquete_actual}.joblib"
            try:
                joblib.dump(paquete_final_a_guardar, nombre_archivo_paquete_final)
                print(f"\nPaquete para '{metodo_actual_features}' ({lag_config_paquete_actual}) guardado como: {nombre_archivo_paquete_final}")
                print(f"Ruta: {os.path.abspath(nombre_archivo_paquete_final)}")
            except Exception as e_dump_final_package:
                print(f"\nERROR al guardar el paquete final para '{metodo_actual_features}': {e_dump_final_package}")
                import traceback
                traceback.print_exc()
        else:
            print(f"\nNo se encontraron modelos válidos para ningún tipo para el método {metodo_actual_features}. No se guardó paquete.")
else:
    print("\nNo se generaron resultados en los experimentos para crear la tabla o guardar paquetes.")

In [ ]:
# Celda: BLOQUE DE PREDICCIÓN PARA DLOCAL (Cargando PAQUETE y Ejecutando MEJORES POR TIPO - v2)

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import joblib
import os # Para os.path.abspath

from sklearn.base import BaseEstimator, TransformerMixin # Por si algún pipeline antiguo la necesita
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline # Por si los objetos guardados son Pipelines

# --- Definiciones de Clases y Funciones Auxiliares ---
class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None): self.columns = columns if columns is not None else []
    def fit(self, X, y=None): return self
    def transform(self, X):
        if not isinstance(X, pd.DataFrame): return X
        return X.copy().drop(columns=self.columns, errors='ignore')

def rmse(y_true, y_pred):
    y_true_np, y_pred_np = np.asarray(y_true), np.asarray(y_pred)
    mask = np.isfinite(y_true_np) & np.isfinite(y_pred_np)
    if not np.all(mask): y_true_np, y_pred_np = y_true_np[mask], y_pred_np[mask]
    if len(y_true_np) == 0: return np.nan
    return np.sqrt(mean_squared_error(y_true_np, y_pred_np))

def evaluate_model(y_test, y_pred):
    y_test_np, y_pred_np = np.asarray(y_test), np.asarray(y_pred)
    mask = np.isfinite(y_test_np) & np.isfinite(y_pred_np)
    if not np.all(mask): y_test_np, y_pred_np = y_test_np[mask], y_pred_np[mask]
    if len(y_test_np) == 0: return {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}
    mae = mean_absolute_error(y_test_np, y_pred_np); rmse_val = rmse(y_test_np, y_pred_np)
    r2 = np.nan
    if len(y_test_np) > 1 and np.var(y_test_np) > 1e-9:
        try: r2 = r2_score(y_test_np, y_pred_np)
        except ValueError: pass
    elif len(y_test_np) > 0 and mean_squared_error(y_test_np, y_pred_np) < 1e-9: r2 = 1.0
    elif len(y_test_np) > 0: r2 = 0.0
    return {'RMSE': rmse_val, 'MAE': mae, 'R2': r2}

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
pd.options.mode.chained_assignment = None

# =====================================================================================
# BLOQUE DE PREDICCIÓN PARA DLOCAL (Carga PAQUETE y Ejecuta MEJORES POR TIPO)
# =====================================================================================

# --- 0. CONFIGURACIÓN Y CARGA DEL PAQUETE DEL MODELO ---
# ASUNCIONES: df_dlocal está cargado.
#             listas_columnas está definida (para obtener columnas_base si no están en el paquete).

nombre_paquete_a_cargar = "paquete_mejores_por_tipo_lineal_mediana__lag1_solo_pred.joblib" # <--- DOBLE GUIÓN BAJO

print(f"Cargando paquete de modelos desde: {nombre_paquete_a_cargar}")
model_package = None
try:
    ruta_paquete = os.path.join(os.getcwd(), nombre_paquete_a_cargar)
    if not os.path.exists(ruta_paquete):
        raise FileNotFoundError(f"El archivo del paquete '{ruta_paquete}' no existe.")
    model_package = joblib.load(ruta_paquete)
    print("Paquete de modelos cargado exitosamente.")

    required_keys = ['metodo_features', 'lag_config_usada', 'log_transform_target', 'mejores_modelos_info_por_tipo']
    if not isinstance(model_package, dict) or not all(key in model_package for key in required_keys):
        print(f"Claves disponibles: {list(model_package.keys()) if isinstance(model_package, dict) else 'No es un dict'}")
        raise ValueError("El archivo cargado no es un paquete de modelos válido o le faltan claves esenciales.")

    metodo_features_paquete = model_package['metodo_features']
    lag_config_paquete = model_package['lag_config_usada']
    log_transform_paquete = model_package['log_transform_target']
    columnas_base_paquete = model_package.get('columnas_base_para_lags')
    mejores_modelos_info = model_package['mejores_modelos_info_por_tipo']

    print(f"  Información del Paquete:")
    print(f"    Método de Features: {metodo_features_paquete}")
    print(f"    Configuración de Lag: {lag_config_paquete}")
    print(f"    Log Transform en Target: {log_transform_paquete}")

    if not columnas_base_paquete:
        if 'listas_columnas' in locals() and metodo_features_paquete in listas_columnas:
            columnas_base_paquete = listas_columnas[metodo_features_paquete]
            print(f"    Columnas Base para Lags (de listas_columnas): {len(columnas_base_paquete)} {columnas_base_paquete[:3]}...")
        else:
            raise ValueError("No se pudieron determinar las 'columnas_base_para_lags'.")
    else:
        print(f"    Columnas Base para Lags (del paquete): {len(columnas_base_paquete)} {columnas_base_paquete[:3]}...")
    print(f"    Contiene mejores modelos para tipos: {list(mejores_modelos_info.keys())}")

except FileNotFoundError:
    raise FileNotFoundError(f"ERROR FATAL: No se encontró el archivo del paquete '{nombre_paquete_a_cargar}'.")
except Exception as e_load_pack:
    raise RuntimeError(f"Error fatal al cargar o procesar el paquete del modelo: {e_load_pack}")

if 'df_dlocal' not in locals() or not isinstance(df_dlocal, pd.DataFrame) or df_dlocal.empty:
     raise NameError("El DataFrame 'df_dlocal' no está definido, está vacío o no es un DataFrame.")

lags_a_generar_dlocal = []
if 'lag1' in lag_config_paquete: lags_a_generar_dlocal.append(1)
if 'lag2' in lag_config_paquete: lags_a_generar_dlocal.append(2)
if 'lag3' in lag_config_paquete: lags_a_generar_dlocal.append(3)
if not lags_a_generar_dlocal:
    raise ValueError(f"No se pudo determinar qué lags numéricos generar a partir de lag_config_paquete: '{lag_config_paquete}'")
print(f"Se generarán lags {lags_a_generar_dlocal} para DLocal.")

print("\n" + "="*80)
print(f"Preparando datos de DLocal para predicción (Método Features: {metodo_features_paquete}, Lags: {lags_a_generar_dlocal})")
print("="*80)

df_dlocal_procesado_base = df_dlocal.copy()

cols_convertir_dlocal = [col for col in columnas_base_paquete if col in df_dlocal_procesado_base.columns and col not in ['Empresa', 'Fecha', 'P_E']]
for col in cols_convertir_dlocal:
    if not pd.api.types.is_numeric_dtype(df_dlocal_procesado_base[col]):
        df_dlocal_procesado_base[col] = pd.to_numeric(df_dlocal_procesado_base[col], errors='coerce')

cols_imputar_dlocal = [col for col in columnas_base_paquete if col in df_dlocal_procesado_base.columns and col not in ['Empresa', 'Fecha', 'P_E']]
if cols_imputar_dlocal:
    df_dlocal_procesado_base.sort_values(['Empresa', 'Fecha'], inplace=True)
    if df_dlocal_procesado_base['Empresa'].nunique() > 1:
        df_dlocal_procesado_base[cols_imputar_dlocal] = df_dlocal_procesado_base.groupby('Empresa')[cols_imputar_dlocal].ffill()
    else:
        df_dlocal_procesado_base[cols_imputar_dlocal] = df_dlocal_procesado_base[cols_imputar_dlocal].ffill()
    df_dlocal_procesado_base[cols_imputar_dlocal] = df_dlocal_procesado_base[cols_imputar_dlocal].fillna(0)

features_base_sin_pe_paquete = [c for c in columnas_base_paquete if c != 'P_E']
features_base_existentes_dlocal = [c for c in features_base_sin_pe_paquete if c in df_dlocal_procesado_base.columns]

df_dlocal_con_todos_lags = df_dlocal_procesado_base[['Empresa', 'Fecha', 'P_E'] + features_base_existentes_dlocal].copy()
all_generated_lag_cols_dlocal = []

for lag_val in lags_a_generar_dlocal:
    for col_base in features_base_existentes_dlocal:
        lag_col_name = f"{col_base}_lag{lag_val}"
        df_dlocal_con_todos_lags[lag_col_name] = df_dlocal_con_todos_lags.groupby('Empresa')[col_base].shift(lag_val)
        all_generated_lag_cols_dlocal.append(lag_col_name)
all_generated_lag_cols_dlocal = sorted(list(set(all_generated_lag_cols_dlocal)))
print(f"Se generaron {len(all_generated_lag_cols_dlocal)} columnas de lag para DLocal: {all_generated_lag_cols_dlocal[:5]}...")

df_dlocal_final_features_con_ids = df_dlocal_con_todos_lags.dropna(subset=all_generated_lag_cols_dlocal).copy()

if df_dlocal_final_features_con_ids.empty:
    raise ValueError("El DataFrame de DLocal (df_dlocal_final_features_con_ids) quedó vacío después de generar lags y dropna.")
print(f"DataFrame DLocal procesado y listo (con IDs y todos los lags): {df_dlocal_final_features_con_ids.shape}")

print("\n" + "="*80)
print("Iterando sobre los mejores modelos por tipo para predicción en DLocal")
print("="*80)

for tipo_modelo_actual, model_info_actual in mejores_modelos_info.items():
    print(f"\n--- Prediciendo con el mejor modelo tipo: {tipo_modelo_actual} ---")

    if model_info_actual is None or 'pipeline_object' not in model_info_actual or model_info_actual['pipeline_object'] is None:
        print(f"  No hay un pipeline válido para el tipo de modelo '{tipo_modelo_actual}'. Omitiendo.")
        continue
    
    pipeline_cargado_actual = model_info_actual['pipeline_object']
    
    estimador_real_actual = pipeline_cargado_actual
    if hasattr(pipeline_cargado_actual, 'steps'):
        estimador_real_actual = pipeline_cargado_actual.steps[-1][1]

    features_esperadas_por_modelo_actual = []
    if hasattr(estimador_real_actual, 'feature_names_in_'):
        features_esperadas_por_modelo_actual = list(estimador_real_actual.feature_names_in_)
        print(f"  Info para {tipo_modelo_actual}: Obtenidas {len(features_esperadas_por_modelo_actual)} features de 'feature_names_in_'.")
    elif hasattr(estimador_real_actual, 'n_features_in_'):
        if estimador_real_actual.n_features_in_ == len(all_generated_lag_cols_dlocal):
            features_esperadas_por_modelo_actual = all_generated_lag_cols_dlocal
            print(f"  Info para {tipo_modelo_actual}: Usando todos los {len(all_generated_lag_cols_dlocal)} lags generados (basado en n_features_in_ coincidente).")
        # MODIFICACIÓN PARA CATBOOST (y otros casos donde n_features_in_ podría ser 0 o no confiable)
        elif (tipo_modelo_actual == 'CB' and estimador_real_actual.n_features_in_ == 0 and len(all_generated_lag_cols_dlocal) > 0) or \
             (estimador_real_actual.n_features_in_ != len(all_generated_lag_cols_dlocal) and len(all_generated_lag_cols_dlocal) > 0) : # Fallback más general
            print(f"  ADVERTENCIA para {tipo_modelo_actual}: n_features_in_ ({estimador_real_actual.n_features_in_}) es 0 o no coincide con lags generados ({len(all_generated_lag_cols_dlocal)}).")
            print(f"  Se asumirá que usa todos los {len(all_generated_lag_cols_dlocal)} lags generados.")
            features_esperadas_por_modelo_actual = all_generated_lag_cols_dlocal
        else:
            print(f"  ERROR para {tipo_modelo_actual}: n_features_in_ ({estimador_real_actual.n_features_in_}) no coincide con lags generados ({len(all_generated_lag_cols_dlocal)}) y no es caso especial. Omitiendo.")
            continue
    else:
        print(f"  ADVERTENCIA CRÍTICA para {tipo_modelo_actual}: No se pueden determinar las features esperadas. Usando TODOS los {len(all_generated_lag_cols_dlocal)} lags generados.")
        features_esperadas_por_modelo_actual = all_generated_lag_cols_dlocal

    if not features_esperadas_por_modelo_actual:
        print(f"  Error: No se pudieron determinar las features para el modelo {tipo_modelo_actual}. Omitiendo.")
        continue
    
    missing_features_in_dlocal = [f for f in features_esperadas_por_modelo_actual if f not in df_dlocal_final_features_con_ids.columns]
    if missing_features_in_dlocal:
        print(f"  ERROR para {tipo_modelo_actual}: Faltan features en DLocal que el modelo espera: {missing_features_in_dlocal}. Omitiendo.")
        continue
        
    X_dlocal_input_this_model = df_dlocal_final_features_con_ids[features_esperadas_por_modelo_actual].copy()
    df_dlocal_eval_data_this_model = df_dlocal_final_features_con_ids[['Fecha', 'P_E']].loc[X_dlocal_input_this_model.index].copy()

    if X_dlocal_input_this_model.empty:
        print(f"  DataFrame de input para el modelo {tipo_modelo_actual} está vacío. Omitiendo.")
        continue
    
    print(f"  Prediciendo con {tipo_modelo_actual} usando {len(features_esperadas_por_modelo_actual)} features: {features_esperadas_por_modelo_actual[:3]}...")

    n_rows_pred_this_model = len(X_dlocal_input_this_model)
    fechas_p, preds_p, reales_p = [], [], []

    if n_rows_pred_this_model > 0:
        if n_rows_pred_this_model > 1:
            print(f"    Realizando {n_rows_pred_this_model-1} predicciones retrospectivas...")
            for i in range(1, n_rows_pred_this_model):
                X_pred_row_loop = X_dlocal_input_this_model.iloc[[i-1]]
                try:
                    pred_raw_loop = pipeline_cargado_actual.predict(X_pred_row_loop)[0]
                    pred_orig_loop = np.expm1(pred_raw_loop) if log_transform_paquete else pred_raw_loop
                    if log_transform_paquete and not np.isfinite(pred_orig_loop): pred_orig_loop = np.nan
                    preds_p.append(pred_orig_loop)
                    fechas_p.append(df_dlocal_eval_data_this_model.iloc[i]['Fecha'])
                    reales_p.append(df_dlocal_eval_data_this_model.iloc[i]['P_E'])
                except Exception as e_pred_loop:
                    print(f"      Error prediciendo para {tipo_modelo_actual} (Fecha: {df_dlocal_eval_data_this_model.iloc[i]['Fecha']}): {e_pred_loop}")
                    preds_p.append(np.nan); fechas_p.append(df_dlocal_eval_data_this_model.iloc[i]['Fecha']); reales_p.append(df_dlocal_eval_data_this_model.iloc[i]['P_E'])
        
        print(f"    Generando forecast para {tipo_modelo_actual}...")
        X_forecast_row_loop = X_dlocal_input_this_model.iloc[[-1]]
        pronostico_final_loop = np.nan
        try:
            pred_raw_forecast_loop = pipeline_cargado_actual.predict(X_forecast_row_loop)[0]
            pronostico_final_loop = np.expm1(pred_raw_forecast_loop) if log_transform_paquete else pred_raw_forecast_loop
            if log_transform_paquete and not np.isfinite(pronostico_final_loop): pronostico_final_loop = np.nan
        except Exception as e_forecast_loop:
            print(f"      Error durante el forecast para {tipo_modelo_actual}: {e_forecast_loop}")
        
        proxima_fecha_loop = "Periodo Siguiente"
        if not df_dlocal_eval_data_this_model.empty and 'Fecha' in df_dlocal_eval_data_this_model.columns and not df_dlocal_eval_data_this_model['Fecha'].empty:
            try: proxima_fecha_loop = df_dlocal_eval_data_this_model['Fecha'].iloc[-1] + pd.DateOffset(months=1)
            except: pass

        df_backtest_resultado_actual = pd.DataFrame({'Fecha': fechas_p, 'P_E_Predicho': preds_p, 'P_E_Real': reales_p})
        df_forecast_resultado_actual = pd.DataFrame({'Fecha': [proxima_fecha_loop], f'P_E_Predicho': [pronostico_final_loop]})

        print(f"\n  --- Resultados del Backtest para DLocal (Modelo Tipo: {tipo_modelo_actual}) ---")
        if not df_backtest_resultado_actual.empty:
            with pd.option_context('display.max_rows', 10): print(df_backtest_resultado_actual.to_string(float_format="%.4f"))
            metricas_actual = evaluate_model(df_backtest_resultado_actual['P_E_Real'], df_backtest_resultado_actual['P_E_Predicho'])
            print(f"  Métricas ({tipo_modelo_actual}): RMSE={metricas_actual['RMSE']:.4f}, MAE={metricas_actual['MAE']:.4f}, R2={metricas_actual['R2']:.4f}")
        else: print("  No hay resultados de backtest.")
        
        print(f"\n  --- Pronóstico para {proxima_fecha_loop} (Modelo Tipo: {tipo_modelo_actual}) ---")
        print(df_forecast_resultado_actual.to_string(float_format="%.4f"))

        if not df_backtest_resultado_actual.empty:
            plt.figure(figsize=(14, 7))
            plt.plot(df_backtest_resultado_actual['Fecha'], df_backtest_resultado_actual['P_E_Real'], marker='.', linestyle='-', label='P/E Real DLocal')
            plt.plot(df_backtest_resultado_actual['Fecha'], df_backtest_resultado_actual['P_E_Predicho'], marker='x', linestyle='--', label=f'P/E Predicho ({tipo_modelo_actual})')
            if isinstance(proxima_fecha_loop, pd.Timestamp) and pd.notna(pronostico_final_loop):
                 plt.scatter([proxima_fecha_loop], [pronostico_final_loop], color='red', label=f'Forecast {tipo_modelo_actual} {proxima_fecha_loop.strftime("%Y-%m")}', zorder=5, s=100)
            plt.title(f'Backtest y Forecast P/E DLocal (Paquete: {metodo_features_paquete}, Modelo: {tipo_modelo_actual}, Lags: {lags_a_generar_dlocal})')
            plt.xlabel('Fecha'); plt.ylabel('P_E Ratio'); plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()
    else:
        print(f"  No hay datos suficientes en DLocal procesado para hacer predicciones con {tipo_modelo_actual}.")

print("\n" + "="*80)
print("Proceso de predicción para DLocal completado para todos los tipos de modelo del paquete.")
print("="*80)

In [ ]:
# Celda: BLOQUE DE PREDICCIÓN PARA DLOCAL (Cargando PAQUETE, Ejecutando MEJORES POR TIPO y ENSAMBLE)

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import joblib
import os # Para os.path.abspath

from sklearn.base import BaseEstimator, TransformerMixin # Por si algún pipeline antiguo la necesita
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline # Por si los objetos guardados son Pipelines

# --- Definiciones de Clases y Funciones Auxiliares ---
class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None): self.columns = columns if columns is not None else []
    def fit(self, X, y=None): return self
    def transform(self, X):
        if not isinstance(X, pd.DataFrame): return X
        return X.copy().drop(columns=self.columns, errors='ignore')

def rmse(y_true, y_pred):
    y_true_np, y_pred_np = np.asarray(y_true), np.asarray(y_pred)
    mask = np.isfinite(y_true_np) & np.isfinite(y_pred_np)
    if not np.all(mask): y_true_np, y_pred_np = y_true_np[mask], y_pred_np[mask]
    if len(y_true_np) == 0: return np.nan
    return np.sqrt(mean_squared_error(y_true_np, y_pred_np))

def evaluate_model(y_test, y_pred):
    y_test_np, y_pred_np = np.asarray(y_test), np.asarray(y_pred)
    mask = np.isfinite(y_test_np) & np.isfinite(y_pred_np)
    if not np.all(mask): y_test_np, y_pred_np = y_test_np[mask], y_pred_np[mask]
    if len(y_test_np) == 0: return {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}
    mae = mean_absolute_error(y_test_np, y_pred_np); rmse_val = rmse(y_test_np, y_pred_np)
    r2 = np.nan
    if len(y_test_np) > 1 and np.var(y_test_np) > 1e-9:
        try: r2 = r2_score(y_test_np, y_pred_np)
        except ValueError: pass
    elif len(y_test_np) > 0 and mean_squared_error(y_test_np, y_pred_np) < 1e-9: r2 = 1.0
    elif len(y_test_np) > 0: r2 = 0.0
    return {'RMSE': rmse_val, 'MAE': mae, 'R2': r2}

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
pd.options.mode.chained_assignment = None

# =====================================================================================
# BLOQUE DE PREDICCIÓN PARA DLOCAL (Carga PAQUETE, Ejecuta MEJORES POR TIPO y ENSAMBLE)
# =====================================================================================

# --- 0. CONFIGURACIÓN Y CARGA DEL PAQUETE DEL MODELO ---
# ASUNCIONES: df_dlocal está cargado.
#             listas_columnas está definida (para obtener columnas_base si no están en el paquete).

nombre_paquete_a_cargar = "paquete_mejores_por_tipo_lineal_mediana__lag1_solo_pred.joblib" # <--- AJUSTA ESTE NOMBRE

print(f"Cargando paquete de modelos desde: {nombre_paquete_a_cargar}")
model_package = None
try:
    ruta_paquete = os.path.join(os.getcwd(), nombre_paquete_a_cargar)
    if not os.path.exists(ruta_paquete):
        raise FileNotFoundError(f"El archivo del paquete '{ruta_paquete}' no existe.")
    model_package = joblib.load(ruta_paquete)
    print("Paquete de modelos cargado exitosamente.")

    required_keys = ['metodo_features', 'lag_config_usada', 'log_transform_target', 'mejores_modelos_info_por_tipo']
    if not isinstance(model_package, dict) or not all(key in model_package for key in required_keys):
        print(f"Claves disponibles: {list(model_package.keys()) if isinstance(model_package, dict) else 'No es un dict'}")
        raise ValueError("El archivo cargado no es un paquete de modelos válido o le faltan claves esenciales.")

    metodo_features_paquete = model_package['metodo_features']
    lag_config_paquete = model_package['lag_config_usada']
    log_transform_paquete = model_package['log_transform_target']
    columnas_base_paquete = model_package.get('columnas_base_para_lags')
    mejores_modelos_info = model_package['mejores_modelos_info_por_tipo']

    print(f"  Información del Paquete:")
    print(f"    Método de Features: {metodo_features_paquete}")
    print(f"    Configuración de Lag: {lag_config_paquete}")
    print(f"    Log Transform en Target: {log_transform_paquete}")

    if not columnas_base_paquete:
        if 'listas_columnas' in locals() and metodo_features_paquete in listas_columnas:
            columnas_base_paquete = listas_columnas[metodo_features_paquete]
            print(f"    Columnas Base para Lags (de listas_columnas): {len(columnas_base_paquete)} {columnas_base_paquete[:3]}...")
        else:
            raise ValueError("No se pudieron determinar las 'columnas_base_para_lags'.")
    else:
        print(f"    Columnas Base para Lags (del paquete): {len(columnas_base_paquete)} {columnas_base_paquete[:3]}...")
    print(f"    Contiene mejores modelos para tipos: {list(mejores_modelos_info.keys())}")

except FileNotFoundError:
    raise FileNotFoundError(f"ERROR FATAL: No se encontró el archivo del paquete '{nombre_paquete_a_cargar}'.")
except Exception as e_load_pack:
    raise RuntimeError(f"Error fatal al cargar o procesar el paquete del modelo: {e_load_pack}")

if 'df_dlocal' not in locals() or not isinstance(df_dlocal, pd.DataFrame) or df_dlocal.empty:
     raise NameError("El DataFrame 'df_dlocal' no está definido, está vacío o no es un DataFrame.")

lags_a_generar_dlocal = []
if 'lag1' in lag_config_paquete: lags_a_generar_dlocal.append(1)
if 'lag2' in lag_config_paquete: lags_a_generar_dlocal.append(2)
if 'lag3' in lag_config_paquete: lags_a_generar_dlocal.append(3)
if not lags_a_generar_dlocal:
    raise ValueError(f"No se pudo determinar qué lags numéricos generar a partir de lag_config_paquete: '{lag_config_paquete}'")
print(f"Se generarán lags {lags_a_generar_dlocal} para DLocal.")

print("\n" + "="*80)
print(f"Preparando datos de DLocal para predicción (Método Features: {metodo_features_paquete}, Lags: {lags_a_generar_dlocal})")
print("="*80)

df_dlocal_procesado_base = df_dlocal.copy()

cols_convertir_dlocal = [col for col in columnas_base_paquete if col in df_dlocal_procesado_base.columns and col not in ['Empresa', 'Fecha', 'P_E']]
for col in cols_convertir_dlocal:
    if not pd.api.types.is_numeric_dtype(df_dlocal_procesado_base[col]):
        df_dlocal_procesado_base[col] = pd.to_numeric(df_dlocal_procesado_base[col], errors='coerce')

cols_imputar_dlocal = [col for col in columnas_base_paquete if col in df_dlocal_procesado_base.columns and col not in ['Empresa', 'Fecha', 'P_E']]
if cols_imputar_dlocal:
    df_dlocal_procesado_base.sort_values(['Empresa', 'Fecha'], inplace=True)
    if df_dlocal_procesado_base['Empresa'].nunique() > 1:
        df_dlocal_procesado_base[cols_imputar_dlocal] = df_dlocal_procesado_base.groupby('Empresa')[cols_imputar_dlocal].ffill()
    else:
        df_dlocal_procesado_base[cols_imputar_dlocal] = df_dlocal_procesado_base[cols_imputar_dlocal].ffill()
    df_dlocal_procesado_base[cols_imputar_dlocal] = df_dlocal_procesado_base[cols_imputar_dlocal].fillna(0)

features_base_sin_pe_paquete = [c for c in columnas_base_paquete if c != 'P_E']
features_base_existentes_dlocal = [c for c in features_base_sin_pe_paquete if c in df_dlocal_procesado_base.columns]

df_dlocal_con_todos_lags = df_dlocal_procesado_base[['Empresa', 'Fecha', 'P_E'] + features_base_existentes_dlocal].copy()
all_generated_lag_cols_dlocal = []

for lag_val in lags_a_generar_dlocal:
    for col_base in features_base_existentes_dlocal:
        lag_col_name = f"{col_base}_lag{lag_val}"
        df_dlocal_con_todos_lags[lag_col_name] = df_dlocal_con_todos_lags.groupby('Empresa')[col_base].shift(lag_val)
        all_generated_lag_cols_dlocal.append(lag_col_name)
all_generated_lag_cols_dlocal = sorted(list(set(all_generated_lag_cols_dlocal)))
print(f"Se generaron {len(all_generated_lag_cols_dlocal)} columnas de lag para DLocal: {all_generated_lag_cols_dlocal[:5]}...")

df_dlocal_final_features_con_ids = df_dlocal_con_todos_lags.dropna(subset=all_generated_lag_cols_dlocal).copy()

if df_dlocal_final_features_con_ids.empty:
    raise ValueError("El DataFrame de DLocal (df_dlocal_final_features_con_ids) quedó vacío después de generar lags y dropna.")
print(f"DataFrame DLocal procesado y listo (con IDs y todos los lags): {df_dlocal_final_features_con_ids.shape}")

# --- 2. ITERAR SOBRE LOS MEJORES MODELOS POR TIPO, PREDECIR Y ALMACENAR PARA ENSAMBLE ---
print("\n" + "="*80)
print("Iterando sobre los mejores modelos por tipo para predicción en DLocal")
print("="*80)

all_backtest_predictions_dict = {}
all_forecast_values_dict = {}
# Estas listas se llenarán una vez con el primer modelo procesado
fechas_comunes_backtest = []
reales_comunes_backtest = []
proxima_fecha_comun = "Periodo Siguiente"


for idx_modelo_tipo, (tipo_modelo_actual, model_info_actual) in enumerate(mejores_modelos_info.items()):
    print(f"\n--- Prediciendo con el mejor modelo tipo: {tipo_modelo_actual} ---")

    if model_info_actual is None or 'pipeline_object' not in model_info_actual or model_info_actual['pipeline_object'] is None:
        print(f"  No hay un pipeline válido para el tipo de modelo '{tipo_modelo_actual}'. Omitiendo.")
        all_backtest_predictions_dict[tipo_modelo_actual] = [] # Guardar lista vacía para mantener consistencia
        all_forecast_values_dict[tipo_modelo_actual] = np.nan
        continue
    
    pipeline_cargado_actual = model_info_actual['pipeline_object']
    
    estimador_real_actual = pipeline_cargado_actual
    if hasattr(pipeline_cargado_actual, 'steps'):
        estimador_real_actual = pipeline_cargado_actual.steps[-1][1]

    features_esperadas_por_modelo_actual = []
    if hasattr(estimador_real_actual, 'feature_names_in_'):
        features_esperadas_por_modelo_actual = list(estimador_real_actual.feature_names_in_)
    elif hasattr(estimador_real_actual, 'n_features_in_'):
        if estimador_real_actual.n_features_in_ == len(all_generated_lag_cols_dlocal):
            features_esperadas_por_modelo_actual = all_generated_lag_cols_dlocal
        elif (tipo_modelo_actual == 'CB' and estimador_real_actual.n_features_in_ == 0 and len(all_generated_lag_cols_dlocal) > 0) or \
             (estimador_real_actual.n_features_in_ != len(all_generated_lag_cols_dlocal) and len(all_generated_lag_cols_dlocal) > 0) :
            features_esperadas_por_modelo_actual = all_generated_lag_cols_dlocal
            print(f"  ADVERTENCIA para {tipo_modelo_actual}: n_features_in_ ({estimador_real_actual.n_features_in_}) es 0 o no coincide. Usando {len(all_generated_lag_cols_dlocal)} lags generados.")
        else:
            print(f"  ERROR para {tipo_modelo_actual}: n_features_in_ ({estimador_real_actual.n_features_in_}) no coincide con lags generados ({len(all_generated_lag_cols_dlocal)}). Omitiendo.")
            all_backtest_predictions_dict[tipo_modelo_actual] = []
            all_forecast_values_dict[tipo_modelo_actual] = np.nan
            continue
    else:
        features_esperadas_por_modelo_actual = all_generated_lag_cols_dlocal
        print(f"  ADVERTENCIA CRÍTICA para {tipo_modelo_actual}: No se determinaron features. Usando TODOS los {len(all_generated_lag_cols_dlocal)} lags generados.")

    if not features_esperadas_por_modelo_actual:
        print(f"  Error: Features esperadas vacía para {tipo_modelo_actual}. Omitiendo.")
        all_backtest_predictions_dict[tipo_modelo_actual] = []
        all_forecast_values_dict[tipo_modelo_actual] = np.nan
        continue
    
    missing_features_in_dlocal = [f for f in features_esperadas_por_modelo_actual if f not in df_dlocal_final_features_con_ids.columns]
    if missing_features_in_dlocal:
        print(f"  ERROR para {tipo_modelo_actual}: Faltan features en DLocal: {missing_features_in_dlocal}. Omitiendo.")
        all_backtest_predictions_dict[tipo_modelo_actual] = []
        all_forecast_values_dict[tipo_modelo_actual] = np.nan
        continue
        
    X_dlocal_input_this_model = df_dlocal_final_features_con_ids[features_esperadas_por_modelo_actual].copy()
    df_dlocal_eval_data_this_model = df_dlocal_final_features_con_ids[['Fecha', 'P_E']].loc[X_dlocal_input_this_model.index].copy()

    if X_dlocal_input_this_model.empty:
        print(f"  DataFrame de input para {tipo_modelo_actual} vacío. Omitiendo.")
        all_backtest_predictions_dict[tipo_modelo_actual] = []
        all_forecast_values_dict[tipo_modelo_actual] = np.nan
        continue
    
    print(f"  Prediciendo con {tipo_modelo_actual} usando {len(features_esperadas_por_modelo_actual)} features...")

    n_rows_pred_this_model = len(X_dlocal_input_this_model)
    current_model_backtest_preds = []
    
    if n_rows_pred_this_model > 0:
        if n_rows_pred_this_model > 1: # Backtest
            for i in range(1, n_rows_pred_this_model):
                X_pred_row_loop = X_dlocal_input_this_model.iloc[[i-1]]
                try:
                    pred_raw_loop = pipeline_cargado_actual.predict(X_pred_row_loop)[0]
                    pred_orig_loop = np.expm1(pred_raw_loop) if log_transform_paquete else pred_raw_loop
                    if log_transform_paquete and not np.isfinite(pred_orig_loop): pred_orig_loop = np.nan
                except Exception as e_pred_loop:
                    pred_orig_loop = np.nan
                    print(f"    Error en backtest para {tipo_modelo_actual} (Fecha target: {df_dlocal_eval_data_this_model.iloc[i]['Fecha']}): {e_pred_loop}")
                current_model_backtest_preds.append(pred_orig_loop)
                if idx_modelo_tipo == 0: # Solo llenar fechas y reales con el primer modelo
                    fechas_comunes_backtest.append(df_dlocal_eval_data_this_model.iloc[i]['Fecha'])
                    reales_comunes_backtest.append(df_dlocal_eval_data_this_model.iloc[i]['P_E'])
        
        all_backtest_predictions_dict[tipo_modelo_actual] = current_model_backtest_preds
        
        # Forecast
        X_forecast_row_loop = X_dlocal_input_this_model.iloc[[-1]]
        pronostico_final_loop = np.nan
        try:
            pred_raw_forecast_loop = pipeline_cargado_actual.predict(X_forecast_row_loop)[0]
            pronostico_final_loop = np.expm1(pred_raw_forecast_loop) if log_transform_paquete else pred_raw_forecast_loop
            if log_transform_paquete and not np.isfinite(pronostico_final_loop): pronostico_final_loop = np.nan
        except Exception as e_forecast_loop:
            print(f"    Error durante el forecast para {tipo_modelo_actual}: {e_forecast_loop}")
        all_forecast_values_dict[tipo_modelo_actual] = pronostico_final_loop
        
        if idx_modelo_tipo == 0 and not df_dlocal_eval_data_this_model.empty : # Calcular proxima_fecha_comun una vez
             if 'Fecha' in df_dlocal_eval_data_this_model.columns and not df_dlocal_eval_data_this_model['Fecha'].empty:
                try: proxima_fecha_comun = df_dlocal_eval_data_this_model['Fecha'].iloc[-1] + pd.DateOffset(months=1)
                except: pass
    else:
        all_backtest_predictions_dict[tipo_modelo_actual] = []
        all_forecast_values_dict[tipo_modelo_actual] = np.nan
        print(f"  No hay datos suficientes en DLocal procesado para hacer predicciones con {tipo_modelo_actual}.")

# --- 3. ENSAMBLE DE PREDICCIONES ---
print("\n" + "="*80)
print("Calculando Ensamble de Predicciones (Promedio Simple)")
print("="*80)

df_ensamble_resultados = pd.DataFrame({'Fecha': fechas_comunes_backtest, 'P_E_Real': reales_comunes_backtest})
num_modelos_validos_ensamble = 0

for tipo_modelo, preds_backtest_modelo in all_backtest_predictions_dict.items():
    if len(preds_backtest_modelo) == len(df_ensamble_resultados):
        df_ensamble_resultados[f'P_E_Pred_{tipo_modelo}'] = preds_backtest_modelo
        num_modelos_validos_ensamble += 1
    else:
        print(f"  Advertencia: Predicciones de backtest para {tipo_modelo} no se incluirán en ensamble (longitud: {len(preds_backtest_modelo)}, esperada: {len(df_ensamble_resultados)}).")

if not df_ensamble_resultados.empty and num_modelos_validos_ensamble > 0:
    cols_predichas_para_ensamble = [col for col in df_ensamble_resultados.columns if col.startswith('P_E_Pred_')]
    df_ensamble_resultados['P_E_Pred_Ensamble'] = df_ensamble_resultados[cols_predichas_para_ensamble].mean(axis=1, skipna=True)

    print("\n--- Resultados del Backtest para DLocal (ENSAMBLE) ---")
    with pd.option_context('display.max_rows', 10): print(df_ensamble_resultados.to_string(float_format="%.4f"))
    
    metricas_ensamble_final = evaluate_model(df_ensamble_resultados['P_E_Real'], df_ensamble_resultados['P_E_Pred_Ensamble'])
    print(f"  Métricas (ENSAMBLE): RMSE={metricas_ensamble_final['RMSE']:.4f}, MAE={metricas_ensamble_final['MAE']:.4f}, R2={metricas_ensamble_final['R2']:.4f}")

    # Ensamblar forecasts
    forecast_values_validos = [val for val in all_forecast_values_dict.values() if pd.notna(val)]
    pronostico_ensamble_final = np.mean(forecast_values_validos) if forecast_values_validos else np.nan
    df_forecast_ensamble_final = pd.DataFrame({'Fecha': [proxima_fecha_comun], 'P_E_Pred_Ensamble': [pronostico_ensamble_final]})
    print(f"\n--- Pronóstico ENSAMBLE para {proxima_fecha_comun} ---")
    print(df_forecast_ensamble_final.to_string(float_format="%.4f"))

    # Graficar Ensamble
    plt.figure(figsize=(15, 8))
    plt.plot(df_ensamble_resultados['Fecha'], df_ensamble_resultados['P_E_Real'], marker='o', linestyle='-', label='P/E Real DLocal', alpha=0.9, linewidth=2)
    plt.plot(df_ensamble_resultados['Fecha'], df_ensamble_resultados['P_E_Pred_Ensamble'], marker='*', linestyle='-', label='P/E Predicho (Ensamble Promedio)', color='purple', linewidth=2, markersize=8)
    
    # Opcional: graficar también los modelos individuales
    for col_pred_individual in cols_predichas_para_ensamble:
        if col_pred_individual != 'P_E_Pred_Ensamble': # No graficar el ensamble dos veces
             plt.plot(df_ensamble_resultados['Fecha'], df_ensamble_resultados[col_pred_individual], linestyle='--', label=col_pred_individual.replace('P_E_Pred_', ''), alpha=0.6)

    if isinstance(proxima_fecha_comun, pd.Timestamp) and pd.notna(pronostico_ensamble_final):
         plt.scatter([proxima_fecha_comun], [pronostico_ensamble_final], color='darkmagenta', marker='*', label=f'Forecast Ensamble {proxima_fecha_comun.strftime("%Y-%m")}', zorder=6, s=150, edgecolors='black')

    plt.title(f'Backtest y Forecast P/E DLocal (Paquete: {metodo_features_paquete}, Lags: {lags_a_generar_dlocal}) - Ensamble y Modelos')
    plt.xlabel('Fecha'); plt.ylabel('P/E Ratio'); plt.legend(loc='upper left', bbox_to_anchor=(1,1)); plt.grid(True); plt.tight_layout(rect=[0, 0, 0.85, 1]); # Ajustar layout para leyenda
    plt.show()
else:
    print("No hay suficientes predicciones válidas para crear o evaluar un ensamble.")

print("\n" + "="*80)
print("Proceso de predicción y ensamble para DLocal completado.")
print("="*80)

#### Con transformación logarítmica

In [ ]:
# Celda: BLOQUE DE EXPERIMENTOS (ENTRENAR SOLO CON LAGS Y GUARDAR PAQUETE POR MÉTODO)

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import joblib
import os # Para os.path.abspath

# --- Modelos ---
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# --- Métricas y Preprocesamiento ---
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer
# from sklearn.base import BaseEstimator, TransformerMixin # No se usa si DropColumns no está en pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, GroupKFold # BaseCrossValidator no se usa directamente
from sklearn.pipeline import Pipeline

# Ignorar warnings comunes (opcional)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
pd.options.mode.chained_assignment = None

# =====================================
# FUNCIONES AUXILIARES (Asegúrate de que estén definidas como en tu script original)
# =====================================
def rmse(y_true, y_pred):
    y_true_np, y_pred_np = np.asarray(y_true), np.asarray(y_pred)
    mask = np.isfinite(y_true_np) & np.isfinite(y_pred_np)
    if not np.all(mask): y_true_np, y_pred_np = y_true_np[mask], y_pred_np[mask]
    if len(y_true_np) == 0: return np.nan
    return np.sqrt(mean_squared_error(y_true_np, y_pred_np))

rmse_scorer = make_scorer(rmse, greater_is_better=False)

def train_test_split_by_empresa(df_model_input, n_empresas_prueba):
    df_model = df_model_input.copy()
    if 'Empresa' not in df_model.columns: raise KeyError("'Empresa' necesaria")
    if 'P_E' not in df_model.columns: raise KeyError("'P_E' (target) necesaria")
    empresas_unicas = df_model['Empresa'].unique()
    n_total_empresas = len(empresas_unicas)
    if n_total_empresas == 0: return None, None, None, None
    if n_total_empresas == 1:
        X = df_model.drop(columns=['P_E']); y = df_model['P_E']
        return X.copy(), y.copy(), pd.DataFrame(columns=X.columns), pd.Series(dtype=y.dtype)
    n_empresas_prueba_valido = max(0, min(n_empresas_prueba, n_total_empresas - 1 if n_total_empresas > 0 else 0))
    if n_empresas_prueba_valido != n_empresas_prueba:
        print(f"  Advertencia train_test_split: n_empresas_prueba ajustado de {n_empresas_prueba} a {n_empresas_prueba_valido}")
    n_empresas_prueba = n_empresas_prueba_valido
    if n_empresas_prueba == 0:
        empresas_entrenamiento = empresas_unicas; empresas_prueba = np.array([])
    else:
        empresas_entrenamiento, empresas_prueba = train_test_split(
            empresas_unicas, test_size=n_empresas_prueba, random_state=42, shuffle=True)
    mask_entrenamiento = df_model['Empresa'].isin(empresas_entrenamiento)
    X = df_model.drop(columns=['P_E']); y = df_model['P_E']
    X_train, y_train = X[mask_entrenamiento].copy(), y[mask_entrenamiento].copy()
    if n_empresas_prueba > 0 and len(empresas_prueba) > 0:
        mask_prueba = df_model['Empresa'].isin(empresas_prueba)
        X_test, y_test = X[mask_prueba].copy(), y[mask_prueba].copy()
        if X_test.empty and not mask_prueba.empty() and not X_train.empty : 
             X_test = pd.DataFrame(columns=X_train.columns)
             y_test = pd.Series(dtype=y_train.dtype)
    else:
        X_test_cols = X_train.columns if not X_train.empty else (X.columns if not X.empty else [])
        y_test_dtype = y_train.dtype if not y_train.empty else (y.dtype if not y.empty else float)
        X_test, y_test = pd.DataFrame(columns=X_test_cols), pd.Series(dtype=y_test_dtype)
    if X_train.empty and n_total_empresas > 0 :
        return None, None, None, None
    return X_train, y_train, X_test, y_test

def evaluate_model(y_test, y_pred):
    y_test_np, y_pred_np = np.asarray(y_test), np.asarray(y_pred)
    mask = np.isfinite(y_test_np) & np.isfinite(y_pred_np)
    if not np.all(mask): y_test_np, y_pred_np = y_test_np[mask], y_pred_np[mask]
    if len(y_test_np) == 0: return {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}
    mae = mean_absolute_error(y_test_np, y_pred_np)
    rmse_val = rmse(y_test_np, y_pred_np)
    r2 = np.nan
    if len(y_test_np) > 1 and np.var(y_test_np) > 1e-9:
        try: r2 = r2_score(y_test_np, y_pred_np)
        except ValueError: pass
    elif len(y_test_np) > 0 and mean_squared_error(y_test_np, y_pred_np) < 1e-9: r2 = 1.0
    elif len(y_test_np) > 0: r2 = 0.0
    return {'RMSE': rmse_val, 'MAE': mae, 'R2': r2}

def sort_by_timestep(X_df, y_series):
    X, y = X_df.copy(), y_series.copy()
    if 'time_step' not in X.columns: return X, y
    if not isinstance(y, pd.Series) or not X.index.equals(y.index):
        if len(y) == len(X): y = pd.Series(np.asarray(y), index=X.index, name=getattr(y, 'name', 'target'))
        else: print("Error sort_by_timestep: y no alineable con X."); return X, y
    X_sorted = X.sort_values('time_step'); y_sorted = y.loc[X_sorted.index]
    return X_sorted, y_sorted

# ================================================================================
# BLOQUE DE EXPERIMENTOS (ENTRENAR SOLO CON CARACTERÍSTICAS REZAGADAS '_lag1')
# ================================================================================
# ASUNCIONES:
# 1. `dfs_shifted`: Diccionario {metodo: DataFrame_con_features_lag1_y_IDs_y_target}
#    Este DataFrame es el resultado del "Bloque 8" que genera los lags.
#    Debe contener 'Empresa', 'Fecha', 'P_E', y las columnas `_lag1` que se usarán como predictores.
#    También puede contener 'time_step' y las columnas base originales (no se usarán como predictores).

results_accumulator = [] # Lista para acumular TODOS los resultados de TODOS los métodos
start_time_total_experiments = time.time()
use_log_transform_experiment = True # Configuración global para este bloque de experimentos
test_size_ratio_experiment = 0.20
LAG_SUFFIX_TO_USE = '_lag1' # Define qué sufijo de lag buscar para los predictores

# --- Iterar sobre los métodos de features ---
for metodo_features, df_original_con_lags_ids_target in dfs_shifted.items():
    start_time_metodo = time.time()
    print("\n" + "="*80)
    print(f"Procesando Método de Features: {metodo_features} {'(CON Log Transform)' if use_log_transform_experiment else '(SIN Log Transform)'} (Predictors: {LAG_SUFFIX_TO_USE})")
    print("="*80 + "\n")

    if df_original_con_lags_ids_target.empty:
        print(f"  ⇨ ADVERTENCIA: DataFrame de entrada para el método {metodo_features} está vacío. Omitiendo.")
        # Podrías añadir una entrada de error a results_accumulator si lo deseas
        continue

    df_current_method_processed = df_original_con_lags_ids_target.copy()

    if 'time_step' not in df_current_method_processed.columns:
        df_current_method_processed.sort_values(['Empresa', 'Fecha'], inplace=True)
        df_current_method_processed['time_step'] = df_current_method_processed.groupby('Empresa').cumcount()

    # Identificar las columnas de predictores (solo las que terminan con el sufijo de lag especificado)
    predictor_cols_for_model = sorted([c for c in df_current_method_processed.columns if c.endswith(LAG_SUFFIX_TO_USE)])
    
    if not predictor_cols_for_model:
        print(f"  ⇨ ADVERTENCIA: No se encontraron columnas '{LAG_SUFFIX_TO_USE}' en dfs_shifted['{metodo_features}']. Omitiendo método.")
        continue
    
    columnas_base_que_generaron_lags = sorted(list(set(lag_col.replace(LAG_SUFFIX_TO_USE, '') for lag_col in predictor_cols_for_model)))
    print(f"  ⇨ Método {metodo_features}: Se usarán {len(predictor_cols_for_model)} predictores ({LAG_SUFFIX_TO_USE}).")
    print(f"    Lags generados de {len(columnas_base_que_generaron_lags)} columnas base: {columnas_base_que_generaron_lags[:5]}...")

    # df_model contendrá IDs, target y SOLO los predictores de lag seleccionados
    cols_for_df_model_construction = ['Empresa', 'Fecha', 'time_step', 'P_E'] + predictor_cols_for_model
    # Asegurar que solo se usen columnas existentes y únicas
    final_cols_for_df_model_construction = sorted(list(set(c for c in cols_for_df_model_construction if c in df_current_method_processed.columns)))
    
    essential_cols_check = ['Empresa', 'Fecha', 'time_step', 'P_E']
    if not all(ec in final_cols_for_df_model_construction for ec in essential_cols_check):
        missing_ess_cols = [ec for ec in essential_cols_check if ec not in final_cols_for_df_model_construction]
        print(f"  ⇨ ERROR: Faltan columnas esenciales ({missing_ess_cols}) para construir df_model para {metodo_features}. Omitiendo.")
        continue
        
    df_model_for_splitting = df_current_method_processed[final_cols_for_df_model_construction].copy()

    # --- Split Train/Test (X_train_with_ids, X_test_with_ids AÚN TIENEN IDs) ---
    n_total_empresas = df_model_for_splitting['Empresa'].nunique()
    n_empresas_test = 0
    if n_total_empresas > 1:
        n_empresas_test = max(1, int(round(n_total_empresas * test_size_ratio_experiment)))
        if n_total_empresas - n_empresas_test < 1 : n_empresas_test = n_total_empresas - 1 # Dejar al menos 1 para entrenar
    
    X_train_with_ids, y_train_original, X_test_with_ids, y_test_original = train_test_split_by_empresa(df_model_for_splitting, n_empresas_test)
    
    if X_train_with_ids is None or X_train_with_ids.empty:
        print(f"  Split train/test fallido o X_train vacío para {metodo_features}. Omitiendo método.")
        continue
    
    # --- Preparar X para el MODELO (SOLO las columnas predictor_cols_for_model) ---
    X_train_model_features = X_train_with_ids[predictor_cols_for_model].copy()
    X_test_model_features = pd.DataFrame(columns=predictor_cols_for_model) # Inicializar vacío
    if X_test_with_ids is not None and not X_test_with_ids.empty:
        X_test_model_features = X_test_with_ids[predictor_cols_for_model].copy()
    
    # --- Transformar Target y Manejar NaNs/Infs ---
    y_train_transformed_target = y_train_original.copy()
    if use_log_transform_experiment:
        y_numeric = pd.to_numeric(y_train_original, errors='coerce')
        # Usar mediana para imputar NaNs en y_train ANTES de log, y asegurar no negativos
        median_y_train = y_numeric.median() if not y_numeric.isnull().all() else 0.0
        y_numeric = y_numeric.fillna(median_y_train)
        y_numeric.loc[y_numeric < 0] = 0 # Evitar errores con log1p
        y_train_transformed_target = np.log1p(y_numeric)
        # Rellenar NaNs que puedan surgir de log1p(valor muy negativo -> -1 -> 0 -> log1p(0)=0)
        # o si y_numeric original era NaN y median_y_train era tal que log1p dio NaN (improbable con fillna(0) para negativos)
        if y_train_transformed_target.isnull().any():
             median_y_transformed = y_train_transformed_target.median() if not y_train_transformed_target.isnull().all() else 0.0
             y_train_transformed_target = y_train_transformed_target.fillna(median_y_transformed)

    # --- Ordenar Datos (X_train_with_ids para CV, X_train_model_features para fit) ---
    # Asegurar alineación de índices después de cualquier manipulación de y_train_transformed_target
    if not X_train_with_ids.index.equals(y_train_transformed_target.index):
        common_idx_train = X_train_with_ids.index.intersection(y_train_transformed_target.index)
        X_train_with_ids = X_train_with_ids.loc[common_idx_train]
        X_train_model_features = X_train_model_features.loc[common_idx_train]
        y_train_transformed_target = y_train_transformed_target.loc[common_idx_train]
        if X_train_model_features.empty : print(f"X_train_model_features vacío tras realinear para {metodo_features}. Omitiendo."); continue

    X_train_cv_sorted, y_train_cv_sorted = sort_by_timestep(X_train_with_ids, y_train_transformed_target)
    X_train_fit_sorted, y_train_fit_sorted = sort_by_timestep(X_train_model_features, y_train_transformed_target) # y_train_fit_sorted es igual a y_train_cv_sorted

    X_test_eval_sorted, y_test_eval_original_sorted = pd.DataFrame(columns=X_test_with_ids.columns), pd.Series(dtype=y_test_original.dtype)
    X_test_predict_sorted = pd.DataFrame(columns=X_test_model_features.columns)
    if X_test_with_ids is not None and not X_test_with_ids.empty:
        X_test_eval_sorted, y_test_eval_original_sorted = sort_by_timestep(X_test_with_ids, y_test_original)
        X_test_predict_sorted, _ = sort_by_timestep(X_test_model_features, y_test_original)


    # --- Grids y Scenarios (igual que tu última versión) ---
    param_grid_rf   = {'model__max_depth':[10, 20, None],'model__min_samples_split':[5, 10],'model__n_estimators':[100, 200]}
    param_grid_xgb  = {'model__n_estimators':[100, 200],'model__max_depth':[3, 5, 7],'model__learning_rate':[0.1, 0.05]}
    param_grid_lgbm = {'model__n_estimators':[100, 200],'model__learning_rate':[0.1, 0.05],'model__max_depth':[3, 5, 7],'model__num_leaves':[15, 31]}
    param_grid_cb   = {'model__iterations':[100, 200],'model__learning_rate':[0.1, 0.05],'model__depth':[3, 5, 7],'model__l2_leaf_reg':[1, 3]}

    scenarios = { # Usar 'model' como prefijo para HP
        'RF_Simple':  {'model': RandomForestRegressor(random_state=42, n_jobs=-1), 'hp': False, 'vt': False, 'prefix': 'model'},
        'RF_HP':      {'model': RandomForestRegressor(random_state=42, n_jobs=-1), 'hp': True,  'vt': False, 'grid': param_grid_rf, 'prefix': 'model'},
        'XGB_Simple': {'model': XGBRegressor(random_state=42, n_jobs=-1), 'hp': False, 'vt': False, 'prefix': 'model'},
        'XGB_HP':     {'model': XGBRegressor(random_state=42, n_jobs=-1), 'hp': True,  'vt': False, 'grid': param_grid_xgb, 'prefix': 'model'},
        'LGBM_Simple':{'model': LGBMRegressor(random_state=42, n_jobs=-1, verbosity=-1), 'hp': False, 'vt': False, 'prefix': 'model'},
        'LGBM_HP':    {'model': LGBMRegressor(random_state=42, n_jobs=-1, verbosity=-1), 'hp': True,  'vt': False, 'grid': param_grid_lgbm, 'prefix': 'model'},
        'CB_Simple':  {'model': CatBoostRegressor(random_state=42, verbose=0, allow_writing_files=False), 'hp': False, 'vt': False, 'prefix': 'model'},
        'CB_HP':      {'model': CatBoostRegressor(random_state=42, verbose=0, allow_writing_files=False), 'hp': True,  'vt': False, 'grid': param_grid_cb, 'prefix': 'model'},
    }
    # Eliminados escenarios _TV y _TVHP para simplificar, ya que vt=True no se estaba usando consistentemente.
    # Si quieres validación temporal (GroupKFold), vt debería ser True.

    # --- Bucle de Escenarios ---
    for escenario_nombre, escenario_cfg in scenarios.items():
        tiempo_esc_inicio = time.time()
        print(f"\n  --- Escenario: {escenario_nombre} para Método: {metodo_features} ---")
        
        current_base_model = escenario_cfg['model']
        perform_hp_search = escenario_cfg['hp']
        use_temporal_cv_in_scenario = escenario_cfg['vt'] # vt=True usaría GroupKFold

        # Datos para CV (pueden tener IDs si use_temporal_cv_in_scenario y GroupKFold)
        X_cv_input_data = X_train_cv_sorted if use_temporal_cv_in_scenario else X_train_with_ids
        y_cv_input_data = y_train_cv_sorted if use_temporal_cv_in_scenario else y_train_transformed_target
        
        # Datos para FIT (siempre sin IDs, solo features de lag)
        X_fit_data = X_train_fit_sorted if use_temporal_cv_in_scenario else X_train_model_features
        y_fit_data = y_train_fit_sorted if use_temporal_cv_in_scenario else y_train_transformed_target

        # Datos para EVALUACIÓN
        X_eval_data_predict = X_test_predict_sorted if use_temporal_cv_in_scenario else X_test_model_features
        y_eval_data_original_final = y_test_eval_original_sorted if use_temporal_cv_in_scenario else y_test_original

        if X_fit_data.empty or y_fit_data.empty:
            print(f"    ERROR: Datos de entrenamiento (X_fit_data o y_fit_data) vacíos para {escenario_nombre}. Omitiendo.")
            # ... (añadir resultado de error a results_accumulator) ...
            results_accumulator.append({
                'Modelo': escenario_nombre.split('_')[0], 'Validacion_Temp': use_temporal_cv_in_scenario,
                'Busqueda_HP': perform_hp_search, 'Config_Key': metodo_features, 'Metodo': metodo_features,
                'Lag_Config': LAG_SUFFIX_TO_USE, 'Log_Transform': use_log_transform_experiment,
                'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan, 'Best_Params': None,
                'pipeline': 'Error: Datos de entrenamiento vacíos', 
                'columnas_base_usadas': columnas_base_que_generaron_lags,
                'n_features_modelo': len(predictor_cols_for_model), 'Error': 'Datos de entrenamiento vacíos'})
            continue

        # Pipeline para GridSearchCV (solo el modelo con prefijo) o modelo base para fit directo
        # NO HAY DropColumns aquí
        pipeline_for_training = Pipeline([(escenario_cfg['prefix'], current_base_model)]) if perform_hp_search else current_base_model
        
        final_fitted_model_object, hp_best_params_found, error_msg_escenario = None, None, None

        try:
            if perform_hp_search:
                cv_strategy_obj, cv_fit_params_dict = None, {}
                can_perform_cv = True
                
                # X_cv_input_data (con IDs) se usa para generar grupos para GroupKFold
                # X_fit_data (sin IDs) se usa para el fit real dentro de GridSearchCV
                n_groups_for_cv = X_cv_input_data['Empresa'].nunique() if use_temporal_cv_in_scenario and 'Empresa' in X_cv_input_data.columns else 0

                if use_temporal_cv_in_scenario and n_groups_for_cv >= 2:
                    groups_for_gkf = X_cv_input_data['Empresa']
                    n_splits_gkf = min(3, n_groups_for_cv); n_splits_gkf = max(2, n_splits_gkf)
                    if len(X_fit_data) < n_splits_gkf : can_perform_cv = False
                    else: cv_strategy_obj = GroupKFold(n_splits=n_splits_gkf); cv_fit_params_dict = {'groups': groups_for_gkf}
                elif len(X_fit_data) >= 2: # KFold estándar
                    n_samples_kf = len(X_fit_data)
                    kfold_splits_val = min(3, n_samples_kf); kfold_splits_val = max(2, kfold_splits_val)
                    if n_samples_kf < kfold_splits_val : can_perform_cv = False
                    else: cv_strategy_obj = KFold(n_splits=kfold_splits_val, shuffle=True, random_state=42)
                else: can_perform_cv = False
                
                if not can_perform_cv or cv_strategy_obj is None:
                    print(f"    Advertencia: No se puede realizar CV para {escenario_nombre}. Se entrenará modelo simple.")
                    perform_hp_search = False # Forzar fit simple
                
                if perform_hp_search: # Re-chequear por si se cambió a False
                    gs = GridSearchCV(pipeline_for_training, escenario_cfg['grid'], scoring=rmse_scorer, 
                                      cv=cv_strategy_obj, n_jobs=-1, refit=True, error_score='raise', verbose=0)
                    print(f"    Iniciando GridSearchCV con {type(cv_strategy_obj).__name__}...")
                    gs.fit(X_fit_data, y_fit_data, **cv_fit_params_dict) # X_fit_data no tiene IDs
                    final_fitted_model_object, hp_best_params_found = gs.best_estimator_, gs.best_params_
                    print(f"    Mejores parámetros: {hp_best_params_found}")

            if not perform_hp_search or final_fitted_model_object is None: # Si HP se omitió o falló
                print(f"    Entrenando modelo simple para {escenario_nombre}...")
                # Si pipeline_for_training era para HP, ahora usamos current_base_model directamente
                model_to_fit_simple = current_base_model 
                model_to_fit_simple.fit(X_fit_data, y_fit_data)
                final_fitted_model_object = model_to_fit_simple
                hp_best_params_found = None # No hubo HP search
        
        except Exception as e:
            error_msg_escenario = str(e)
            print(f"    ERROR durante entrenamiento/HP search para {escenario_nombre}: {e}")

        # --- Evaluación ---
        eval_metrics_dict, n_features_in_final_model = {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}, "N/A"
        if final_fitted_model_object:
            try:
                # Obtener el estimador real del pipeline (si es un pipeline de GridSearchCV) o el modelo mismo
                actual_estimator = final_fitted_model_object
                if hasattr(final_fitted_model_object, 'steps'): # Es un Pipeline
                    actual_estimator = final_fitted_model_object.steps[-1][1]
                
                n_features_in_final_model = getattr(actual_estimator, 'n_features_in_', "N/A")
                # print(f"    >> Estimador final '{type(actual_estimator).__name__}' vio {n_features_in_final_model} features.")
                # if isinstance(n_features_in_final_model, int) and n_features_in_final_model != len(predictor_cols_for_model):
                #      print(f"       ¡ADVERTENCIA! n_features ({n_features_in_final_model}) no coincide con num. de lags ({len(predictor_cols_for_model)}).")

                if X_eval_data_predict is not None and not X_eval_data_predict.empty and \
                   y_eval_data_original_final is not None and not y_eval_data_original_final.empty:
                    
                    pred_transformed_eval = final_fitted_model_object.predict(X_eval_data_predict)
                    pred_original_scale_eval = np.expm1(pred_transformed_eval) if use_log_transform_experiment else pred_transformed_eval
                    if use_log_transform_experiment: pred_original_scale_eval[~np.isfinite(pred_original_scale_eval)] = np.nan # Manejar Infs de expm1
                    
                    eval_metrics_dict = evaluate_model(y_eval_data_original_final, pred_original_scale_eval)
                    print(f"    Métricas Test {escenario_nombre}: RMSE={eval_metrics_dict['RMSE']:.4f}, MAE={eval_metrics_dict['MAE']:.4f}, R2={eval_metrics_dict['R2']:.4f}")
                else:
                    print(f"    No hay datos de X_eval ({X_eval_data_predict.shape if X_eval_data_predict is not None else 'None'}) / y_eval ({y_eval_data_original_final.shape if y_eval_data_original_final is not None else 'None'}) para evaluar.")
            except Exception as e_eval:
                error_msg_escenario = (error_msg_escenario + f" | EvalError: {str(e_eval)}") if error_msg_escenario else f"EvalError: {str(e_eval)}"
                print(f"    ERROR durante evaluación para {escenario_nombre}: {e_eval}")
        
        results_accumulator.append({
            'Modelo': escenario_nombre.split('_')[0], 
            'Validacion_Temp': use_temporal_cv_in_scenario, # Si se usó GroupKFold
            'Busqueda_HP': escenario_cfg['hp'] and perform_hp_search, # Si HP se intentó y se completó
            'Config_Key': metodo_features, 
            'Metodo': metodo_features,
            'Lag_Config': LAG_SUFFIX_TO_USE + '_solo_pred', 
            'Log_Transform': use_log_transform_experiment, 
            'RMSE': eval_metrics_dict['RMSE'],
            'MAE': eval_metrics_dict['MAE'], 
            'R2': eval_metrics_dict['R2'], 
            'Best_Params': hp_best_params_found,
            'pipeline': final_fitted_model_object, # El objeto modelo/pipeline entrenado
            'columnas_base_usadas': columnas_base_que_generaron_lags,
            'n_features_modelo': n_features_in_final_model, 
            'Error': error_msg_escenario
        })
        print(f"    Tiempo escenario {escenario_nombre}: {time.time() - tiempo_esc_inicio:.2f}s")
    # --- Fin del bucle de escenarios ---
    print(f"\nTiempo total para Método de Features {metodo_features}: {time.time() - start_time_metodo:.2f}s")
# --- Fin del bucle de métodos de features ---
print(f"\n\nTiempo total de ejecución de todos los experimentos: {(time.time() - start_time_total_experiments)/60:.2f} minutos")


# ================================================================================
# CREAR TABLA DE RESULTADOS GENERAL Y GUARDAR PAQUETES POR MÉTODO
# ================================================================================
if results_accumulator:
    df_results_all_experiments = pd.DataFrame(results_accumulator)
    print("\n" + "="*80 + "\n--- Resultados Finales Globales (Modelos Entrenados SOLO CON LAGS) ---\n" + f"Total resultados generados: {len(df_results_all_experiments)}\n")
    
    cols_to_display_summary = ['Metodo', 'Modelo', 'Busqueda_HP', 'RMSE', 'MAE', 'R2', 'n_features_modelo', 'Error', 'Best_Params']
    final_cols_to_display_summary = [col for col in cols_to_display_summary if col in df_results_all_experiments.columns]
    with pd.option_context('display.max_rows', 200, 'display.max_columns', None, 'display.width', 2000, 'display.float_format', '{:.4f}'.format):
        print(df_results_all_experiments.sort_values(by=['Metodo','RMSE'], ascending=[True,True])[final_cols_to_display_summary].to_string(index=False))

    # --- GUARDAR PAQUETE CON MEJORES MODELOS POR TIPO PARA CADA MÉTODO DE FEATURES ---
    for metodo_actual_features in df_results_all_experiments['Metodo'].unique():
        print("\n" + "="*70)
        print(f"--- Preparando paquete de mejores modelos por tipo para Método de Features: {metodo_actual_features} ---")
        
        df_resultados_metodo_actual = df_results_all_experiments[
            (df_results_all_experiments['Metodo'] == metodo_actual_features) &
            (df_results_all_experiments['RMSE'].notna()) &
            (df_results_all_experiments['pipeline'].notna()) & 
            (df_results_all_experiments['Error'].isna()) # Solo resultados sin errores de entrenamiento/eval
        ].copy()

        if df_resultados_metodo_actual.empty:
            print(f"No hay resultados válidos para el método '{metodo_actual_features}' para crear un paquete.")
            continue

        mejores_modelos_info_por_tipo = {}
        modelos_base_a_considerar = ['RF', 'XGB', 'LGBM', 'CB']
        
        # Tomar metadata del primer resultado válido para este método
        # (asumiendo que Log_Transform, columnas_base_usadas, Lag_Config son consistentes para el método)
        metadata_ref_row = df_resultados_metodo_actual.iloc[0]
        log_transform_paquete_actual = metadata_ref_row.get('Log_Transform', True)
        columnas_base_paquete_actual = metadata_ref_row.get('columnas_base_usadas', [])
        lag_config_paquete_actual = metadata_ref_row.get('Lag_Config', LAG_SUFFIX_TO_USE + '_solo_pred')


        for tipo_modelo in modelos_base_a_considerar:
            df_tipo_especifico = df_resultados_metodo_actual[df_resultados_metodo_actual['Modelo'] == tipo_modelo]
            
            if not df_tipo_especifico.empty:
                mejor_fila_tipo_actual = df_tipo_especifico.sort_values(by='RMSE', ascending=True).iloc[0]
                
                mejores_modelos_info_por_tipo[tipo_modelo] = {
                    'pipeline_object': mejor_fila_tipo_actual['pipeline'], # El modelo/pipeline entrenado
                    'RMSE': mejor_fila_tipo_actual['RMSE'],
                    'MAE': mejor_fila_tipo_actual['MAE'],
                    'R2': mejor_fila_tipo_actual['R2'],
                    'Best_Params': mejor_fila_tipo_actual['Best_Params'],
                    'n_features_modelo': mejor_fila_tipo_actual['n_features_modelo'],
                    'Escenario_Validacion_Temp': mejor_fila_tipo_actual['Validacion_Temp'],
                    'Escenario_Busqueda_HP': mejor_fila_tipo_actual['Busqueda_HP']
                }
                print(f"  Mejor {tipo_modelo} para {metodo_actual_features}: RMSE = {mejor_fila_tipo_actual['RMSE']:.4f}, n_features={mejor_fila_tipo_actual['n_features_modelo']}")
            else:
                print(f"  No se encontraron modelos válidos para {tipo_modelo} para el método {metodo_actual_features}.")
                mejores_modelos_info_por_tipo[tipo_modelo] = None 

        if any(mejores_modelos_info_por_tipo.values()): # Si se encontró al menos un mejor modelo
            paquete_final_a_guardar = {
                'metodo_features': metodo_actual_features,
                'lag_config_usada': lag_config_paquete_actual,
                'log_transform_target': log_transform_paquete_actual,
                'columnas_base_para_lags': columnas_base_paquete_actual,
                'mejores_modelos_info_por_tipo': mejores_modelos_info_por_tipo
            }
            
            nombre_archivo_paquete_final = f"paquete_mejores_por_tipo_{metodo_actual_features.replace(' ', '_')}_{lag_config_paquete_actual}.joblib"
            try:
                joblib.dump(paquete_final_a_guardar, nombre_archivo_paquete_final)
                print(f"\nPaquete para '{metodo_actual_features}' ({lag_config_paquete_actual}) guardado como: {nombre_archivo_paquete_final}")
                print(f"Ruta: {os.path.abspath(nombre_archivo_paquete_final)}")
            except Exception as e_dump_final_package:
                print(f"\nERROR al guardar el paquete final para '{metodo_actual_features}': {e_dump_final_package}")
                import traceback
                traceback.print_exc()
        else:
            print(f"\nNo se encontraron modelos válidos para ningún tipo para el método {metodo_actual_features}. No se guardó paquete.")
else:
    print("\nNo se generaron resultados en los experimentos para crear la tabla o guardar paquetes.")

In [ ]:
# Celda: BLOQUE DE PREDICCIÓN PARA DLOCAL (Cargando PAQUETE y Ejecutando MEJORES POR TIPO - v2)

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import joblib
import os # Para os.path.abspath

from sklearn.base import BaseEstimator, TransformerMixin # Por si algún pipeline antiguo la necesita
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline # Por si los objetos guardados son Pipelines

# --- Definiciones de Clases y Funciones Auxiliares ---
class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None): self.columns = columns if columns is not None else []
    def fit(self, X, y=None): return self
    def transform(self, X):
        if not isinstance(X, pd.DataFrame): return X
        return X.copy().drop(columns=self.columns, errors='ignore')

def rmse(y_true, y_pred):
    y_true_np, y_pred_np = np.asarray(y_true), np.asarray(y_pred)
    mask = np.isfinite(y_true_np) & np.isfinite(y_pred_np)
    if not np.all(mask): y_true_np, y_pred_np = y_true_np[mask], y_pred_np[mask]
    if len(y_true_np) == 0: return np.nan
    return np.sqrt(mean_squared_error(y_true_np, y_pred_np))

def evaluate_model(y_test, y_pred):
    y_test_np, y_pred_np = np.asarray(y_test), np.asarray(y_pred)
    mask = np.isfinite(y_test_np) & np.isfinite(y_pred_np)
    if not np.all(mask): y_test_np, y_pred_np = y_test_np[mask], y_pred_np[mask]
    if len(y_test_np) == 0: return {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}
    mae = mean_absolute_error(y_test_np, y_pred_np); rmse_val = rmse(y_test_np, y_pred_np)
    r2 = np.nan
    if len(y_test_np) > 1 and np.var(y_test_np) > 1e-9:
        try: r2 = r2_score(y_test_np, y_pred_np)
        except ValueError: pass
    elif len(y_test_np) > 0 and mean_squared_error(y_test_np, y_pred_np) < 1e-9: r2 = 1.0
    elif len(y_test_np) > 0: r2 = 0.0
    return {'RMSE': rmse_val, 'MAE': mae, 'R2': r2}

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
pd.options.mode.chained_assignment = None

# =====================================================================================
# BLOQUE DE PREDICCIÓN PARA DLOCAL (Carga PAQUETE y Ejecuta MEJORES POR TIPO)
# =====================================================================================

# --- 0. CONFIGURACIÓN Y CARGA DEL PAQUETE DEL MODELO ---
# ASUNCIONES: df_dlocal está cargado.
#             listas_columnas está definida (para obtener columnas_base si no están en el paquete).

nombre_paquete_a_cargar = "paquete_mejores_por_tipo_lineal_mediana__lag1_solo_pred.joblib" # <--- DOBLE GUIÓN BAJO

print(f"Cargando paquete de modelos desde: {nombre_paquete_a_cargar}")
model_package = None
try:
    ruta_paquete = os.path.join(os.getcwd(), nombre_paquete_a_cargar)
    if not os.path.exists(ruta_paquete):
        raise FileNotFoundError(f"El archivo del paquete '{ruta_paquete}' no existe.")
    model_package = joblib.load(ruta_paquete)
    print("Paquete de modelos cargado exitosamente.")

    required_keys = ['metodo_features', 'lag_config_usada', 'log_transform_target', 'mejores_modelos_info_por_tipo']
    if not isinstance(model_package, dict) or not all(key in model_package for key in required_keys):
        print(f"Claves disponibles: {list(model_package.keys()) if isinstance(model_package, dict) else 'No es un dict'}")
        raise ValueError("El archivo cargado no es un paquete de modelos válido o le faltan claves esenciales.")

    metodo_features_paquete = model_package['metodo_features']
    lag_config_paquete = model_package['lag_config_usada']
    log_transform_paquete = model_package['log_transform_target']
    columnas_base_paquete = model_package.get('columnas_base_para_lags')
    mejores_modelos_info = model_package['mejores_modelos_info_por_tipo']

    print(f"  Información del Paquete:")
    print(f"    Método de Features: {metodo_features_paquete}")
    print(f"    Configuración de Lag: {lag_config_paquete}")
    print(f"    Log Transform en Target: {log_transform_paquete}")

    if not columnas_base_paquete:
        if 'listas_columnas' in locals() and metodo_features_paquete in listas_columnas:
            columnas_base_paquete = listas_columnas[metodo_features_paquete]
            print(f"    Columnas Base para Lags (de listas_columnas): {len(columnas_base_paquete)} {columnas_base_paquete[:3]}...")
        else:
            raise ValueError("No se pudieron determinar las 'columnas_base_para_lags'.")
    else:
        print(f"    Columnas Base para Lags (del paquete): {len(columnas_base_paquete)} {columnas_base_paquete[:3]}...")
    print(f"    Contiene mejores modelos para tipos: {list(mejores_modelos_info.keys())}")

except FileNotFoundError:
    raise FileNotFoundError(f"ERROR FATAL: No se encontró el archivo del paquete '{nombre_paquete_a_cargar}'.")
except Exception as e_load_pack:
    raise RuntimeError(f"Error fatal al cargar o procesar el paquete del modelo: {e_load_pack}")

if 'df_dlocal' not in locals() or not isinstance(df_dlocal, pd.DataFrame) or df_dlocal.empty:
     raise NameError("El DataFrame 'df_dlocal' no está definido, está vacío o no es un DataFrame.")

lags_a_generar_dlocal = []
if 'lag1' in lag_config_paquete: lags_a_generar_dlocal.append(1)
if 'lag2' in lag_config_paquete: lags_a_generar_dlocal.append(2)
if 'lag3' in lag_config_paquete: lags_a_generar_dlocal.append(3)
if not lags_a_generar_dlocal:
    raise ValueError(f"No se pudo determinar qué lags numéricos generar a partir de lag_config_paquete: '{lag_config_paquete}'")
print(f"Se generarán lags {lags_a_generar_dlocal} para DLocal.")

print("\n" + "="*80)
print(f"Preparando datos de DLocal para predicción (Método Features: {metodo_features_paquete}, Lags: {lags_a_generar_dlocal})")
print("="*80)

df_dlocal_procesado_base = df_dlocal.copy()

cols_convertir_dlocal = [col for col in columnas_base_paquete if col in df_dlocal_procesado_base.columns and col not in ['Empresa', 'Fecha', 'P_E']]
for col in cols_convertir_dlocal:
    if not pd.api.types.is_numeric_dtype(df_dlocal_procesado_base[col]):
        df_dlocal_procesado_base[col] = pd.to_numeric(df_dlocal_procesado_base[col], errors='coerce')

cols_imputar_dlocal = [col for col in columnas_base_paquete if col in df_dlocal_procesado_base.columns and col not in ['Empresa', 'Fecha', 'P_E']]
if cols_imputar_dlocal:
    df_dlocal_procesado_base.sort_values(['Empresa', 'Fecha'], inplace=True)
    if df_dlocal_procesado_base['Empresa'].nunique() > 1:
        df_dlocal_procesado_base[cols_imputar_dlocal] = df_dlocal_procesado_base.groupby('Empresa')[cols_imputar_dlocal].ffill()
    else:
        df_dlocal_procesado_base[cols_imputar_dlocal] = df_dlocal_procesado_base[cols_imputar_dlocal].ffill()
    df_dlocal_procesado_base[cols_imputar_dlocal] = df_dlocal_procesado_base[cols_imputar_dlocal].fillna(0)

features_base_sin_pe_paquete = [c for c in columnas_base_paquete if c != 'P_E']
features_base_existentes_dlocal = [c for c in features_base_sin_pe_paquete if c in df_dlocal_procesado_base.columns]

df_dlocal_con_todos_lags = df_dlocal_procesado_base[['Empresa', 'Fecha', 'P_E'] + features_base_existentes_dlocal].copy()
all_generated_lag_cols_dlocal = []

for lag_val in lags_a_generar_dlocal:
    for col_base in features_base_existentes_dlocal:
        lag_col_name = f"{col_base}_lag{lag_val}"
        df_dlocal_con_todos_lags[lag_col_name] = df_dlocal_con_todos_lags.groupby('Empresa')[col_base].shift(lag_val)
        all_generated_lag_cols_dlocal.append(lag_col_name)
all_generated_lag_cols_dlocal = sorted(list(set(all_generated_lag_cols_dlocal)))
print(f"Se generaron {len(all_generated_lag_cols_dlocal)} columnas de lag para DLocal: {all_generated_lag_cols_dlocal[:5]}...")

df_dlocal_final_features_con_ids = df_dlocal_con_todos_lags.dropna(subset=all_generated_lag_cols_dlocal).copy()

if df_dlocal_final_features_con_ids.empty:
    raise ValueError("El DataFrame de DLocal (df_dlocal_final_features_con_ids) quedó vacío después de generar lags y dropna.")
print(f"DataFrame DLocal procesado y listo (con IDs y todos los lags): {df_dlocal_final_features_con_ids.shape}")

print("\n" + "="*80)
print("Iterando sobre los mejores modelos por tipo para predicción en DLocal")
print("="*80)

for tipo_modelo_actual, model_info_actual in mejores_modelos_info.items():
    print(f"\n--- Prediciendo con el mejor modelo tipo: {tipo_modelo_actual} ---")

    if model_info_actual is None or 'pipeline_object' not in model_info_actual or model_info_actual['pipeline_object'] is None:
        print(f"  No hay un pipeline válido para el tipo de modelo '{tipo_modelo_actual}'. Omitiendo.")
        continue
    
    pipeline_cargado_actual = model_info_actual['pipeline_object']
    
    estimador_real_actual = pipeline_cargado_actual
    if hasattr(pipeline_cargado_actual, 'steps'):
        estimador_real_actual = pipeline_cargado_actual.steps[-1][1]

    features_esperadas_por_modelo_actual = []
    if hasattr(estimador_real_actual, 'feature_names_in_'):
        features_esperadas_por_modelo_actual = list(estimador_real_actual.feature_names_in_)
        print(f"  Info para {tipo_modelo_actual}: Obtenidas {len(features_esperadas_por_modelo_actual)} features de 'feature_names_in_'.")
    elif hasattr(estimador_real_actual, 'n_features_in_'):
        if estimador_real_actual.n_features_in_ == len(all_generated_lag_cols_dlocal):
            features_esperadas_por_modelo_actual = all_generated_lag_cols_dlocal
            print(f"  Info para {tipo_modelo_actual}: Usando todos los {len(all_generated_lag_cols_dlocal)} lags generados (basado en n_features_in_ coincidente).")
        # MODIFICACIÓN PARA CATBOOST (y otros casos donde n_features_in_ podría ser 0 o no confiable)
        elif (tipo_modelo_actual == 'CB' and estimador_real_actual.n_features_in_ == 0 and len(all_generated_lag_cols_dlocal) > 0) or \
             (estimador_real_actual.n_features_in_ != len(all_generated_lag_cols_dlocal) and len(all_generated_lag_cols_dlocal) > 0) : # Fallback más general
            print(f"  ADVERTENCIA para {tipo_modelo_actual}: n_features_in_ ({estimador_real_actual.n_features_in_}) es 0 o no coincide con lags generados ({len(all_generated_lag_cols_dlocal)}).")
            print(f"  Se asumirá que usa todos los {len(all_generated_lag_cols_dlocal)} lags generados.")
            features_esperadas_por_modelo_actual = all_generated_lag_cols_dlocal
        else:
            print(f"  ERROR para {tipo_modelo_actual}: n_features_in_ ({estimador_real_actual.n_features_in_}) no coincide con lags generados ({len(all_generated_lag_cols_dlocal)}) y no es caso especial. Omitiendo.")
            continue
    else:
        print(f"  ADVERTENCIA CRÍTICA para {tipo_modelo_actual}: No se pueden determinar las features esperadas. Usando TODOS los {len(all_generated_lag_cols_dlocal)} lags generados.")
        features_esperadas_por_modelo_actual = all_generated_lag_cols_dlocal

    if not features_esperadas_por_modelo_actual:
        print(f"  Error: No se pudieron determinar las features para el modelo {tipo_modelo_actual}. Omitiendo.")
        continue
    
    missing_features_in_dlocal = [f for f in features_esperadas_por_modelo_actual if f not in df_dlocal_final_features_con_ids.columns]
    if missing_features_in_dlocal:
        print(f"  ERROR para {tipo_modelo_actual}: Faltan features en DLocal que el modelo espera: {missing_features_in_dlocal}. Omitiendo.")
        continue
        
    X_dlocal_input_this_model = df_dlocal_final_features_con_ids[features_esperadas_por_modelo_actual].copy()
    df_dlocal_eval_data_this_model = df_dlocal_final_features_con_ids[['Fecha', 'P_E']].loc[X_dlocal_input_this_model.index].copy()

    if X_dlocal_input_this_model.empty:
        print(f"  DataFrame de input para el modelo {tipo_modelo_actual} está vacío. Omitiendo.")
        continue
    
    print(f"  Prediciendo con {tipo_modelo_actual} usando {len(features_esperadas_por_modelo_actual)} features: {features_esperadas_por_modelo_actual[:3]}...")

    n_rows_pred_this_model = len(X_dlocal_input_this_model)
    fechas_p, preds_p, reales_p = [], [], []

    if n_rows_pred_this_model > 0:
        if n_rows_pred_this_model > 1:
            print(f"    Realizando {n_rows_pred_this_model-1} predicciones retrospectivas...")
            for i in range(1, n_rows_pred_this_model):
                X_pred_row_loop = X_dlocal_input_this_model.iloc[[i-1]]
                try:
                    pred_raw_loop = pipeline_cargado_actual.predict(X_pred_row_loop)[0]
                    pred_orig_loop = np.expm1(pred_raw_loop) if log_transform_paquete else pred_raw_loop
                    if log_transform_paquete and not np.isfinite(pred_orig_loop): pred_orig_loop = np.nan
                    preds_p.append(pred_orig_loop)
                    fechas_p.append(df_dlocal_eval_data_this_model.iloc[i]['Fecha'])
                    reales_p.append(df_dlocal_eval_data_this_model.iloc[i]['P_E'])
                except Exception as e_pred_loop:
                    print(f"      Error prediciendo para {tipo_modelo_actual} (Fecha: {df_dlocal_eval_data_this_model.iloc[i]['Fecha']}): {e_pred_loop}")
                    preds_p.append(np.nan); fechas_p.append(df_dlocal_eval_data_this_model.iloc[i]['Fecha']); reales_p.append(df_dlocal_eval_data_this_model.iloc[i]['P_E'])
        
        print(f"    Generando forecast para {tipo_modelo_actual}...")
        X_forecast_row_loop = X_dlocal_input_this_model.iloc[[-1]]
        pronostico_final_loop = np.nan
        try:
            pred_raw_forecast_loop = pipeline_cargado_actual.predict(X_forecast_row_loop)[0]
            pronostico_final_loop = np.expm1(pred_raw_forecast_loop) if log_transform_paquete else pred_raw_forecast_loop
            if log_transform_paquete and not np.isfinite(pronostico_final_loop): pronostico_final_loop = np.nan
        except Exception as e_forecast_loop:
            print(f"      Error durante el forecast para {tipo_modelo_actual}: {e_forecast_loop}")
        
        proxima_fecha_loop = "Periodo Siguiente"
        if not df_dlocal_eval_data_this_model.empty and 'Fecha' in df_dlocal_eval_data_this_model.columns and not df_dlocal_eval_data_this_model['Fecha'].empty:
            try: proxima_fecha_loop = df_dlocal_eval_data_this_model['Fecha'].iloc[-1] + pd.DateOffset(months=1)
            except: pass

        df_backtest_resultado_actual = pd.DataFrame({'Fecha': fechas_p, 'P_E_Predicho': preds_p, 'P_E_Real': reales_p})
        df_forecast_resultado_actual = pd.DataFrame({'Fecha': [proxima_fecha_loop], f'P_E_Predicho': [pronostico_final_loop]})

        print(f"\n  --- Resultados del Backtest para DLocal (Modelo Tipo: {tipo_modelo_actual}) ---")
        if not df_backtest_resultado_actual.empty:
            with pd.option_context('display.max_rows', 10): print(df_backtest_resultado_actual.to_string(float_format="%.4f"))
            metricas_actual = evaluate_model(df_backtest_resultado_actual['P_E_Real'], df_backtest_resultado_actual['P_E_Predicho'])
            print(f"  Métricas ({tipo_modelo_actual}): RMSE={metricas_actual['RMSE']:.4f}, MAE={metricas_actual['MAE']:.4f}, R2={metricas_actual['R2']:.4f}")
        else: print("  No hay resultados de backtest.")
        
        print(f"\n  --- Pronóstico para {proxima_fecha_loop} (Modelo Tipo: {tipo_modelo_actual}) ---")
        print(df_forecast_resultado_actual.to_string(float_format="%.4f"))

        if not df_backtest_resultado_actual.empty:
            plt.figure(figsize=(14, 7))
            plt.plot(df_backtest_resultado_actual['Fecha'], df_backtest_resultado_actual['P_E_Real'], marker='.', linestyle='-', label='P/E Real DLocal')
            plt.plot(df_backtest_resultado_actual['Fecha'], df_backtest_resultado_actual['P_E_Predicho'], marker='x', linestyle='--', label=f'P/E Predicho ({tipo_modelo_actual})')
            if isinstance(proxima_fecha_loop, pd.Timestamp) and pd.notna(pronostico_final_loop):
                 plt.scatter([proxima_fecha_loop], [pronostico_final_loop], color='red', label=f'Forecast {tipo_modelo_actual} {proxima_fecha_loop.strftime("%Y-%m")}', zorder=5, s=100)
            plt.title(f'Backtest y Forecast P/E DLocal (Paquete: {metodo_features_paquete}, Modelo: {tipo_modelo_actual}, Lags: {lags_a_generar_dlocal})')
            plt.xlabel('Fecha'); plt.ylabel('P_E Ratio'); plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()
    else:
        print(f"  No hay datos suficientes en DLocal procesado para hacer predicciones con {tipo_modelo_actual}.")

print("\n" + "="*80)
print("Proceso de predicción para DLocal completado para todos los tipos de modelo del paquete.")
print("="*80)